# Prepare

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys 
expts_path = 'C:\\Users\\Administrator\\Documents\\multimode_expts_tprocv2'

# Add the path to the system path at the highest priority
sys.path.insert(0, expts_path)
print('Path added at highest priority')

# Verify the path is added
print('Path added:')
print(sys.path)

# Import the experiments module from multimode
import experiments as meas

# # Verify the module is imported from the correct path
print('Experiments module path:')
print(meas.__file__)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from copy import deepcopy

import experiments as meas
from meas_utils import MultimodeStation, CharacterizationRunner
from slab import AttrDict
print('going into mm station')
station = MultimodeStation(
    experiment_name = "2600403_lmm",
    hardware_config ='hardware_config_20260111.yml'
    # experiment_name = "251111_qec",
)

cfg_dict = station.cfg_dict
config_thisrun = station.config_thisrun

In [ ]:
meas = station.meas

## Import

### Previous experiment data

Handling of parameter updating: 

The experiment objects are initialized with the `config_path`, meaning they will read from our hardware config file and set the `self.cfg` to that. 
But here we will be overriding that immediately with a new config AttrDict that is at first just a deepcopy of that yaml file but as we go over each cell the values inside this `config_thisrun` will be gradually updated. 
Throughout the notebook, the updated config will only live in memory, until you decide to write it to disk.


What experiments to run?
 
Depending on stage of cooldown, we will run a different sequence of calibration experiments. For example, amplitude rabi don't need to be updated every time, but the frequency correction from T2 is important to do every day. In the dictionary experiments to run, we will speciy the experiments we want to run. 

## Datset for Sidebands

## Config

# Experiments to run

In [ ]:
expts_to_run = {# readout 
                'res_spec': True, # Readout spectroscopy
                'single_shot': True, 
                # qubit ge 
                'pulse_probe_ge': True,
                't2_ge': True, 
                'amplitude_ge': True,
                't1_ge': True,
                # qubit ef
                'pulse_probe_ef': True,
                't2_ef': True,
                'amplitude_ef': True,
                't1_ef': True,

                # manipulate 
                'man_modes': [1], # [1,2] if also want to run mode 2
                'pulse_probe_f0g1': True,
                'length_rabi_sweep':True,
                'length_rabi':False, # this will run automatically if the length_rabi_sweep is set to True
                'chi_ge': True, 
                'chi_ef': True,
                'RB': False,

                #storage
                'stor_modes': [1,2,3,4,5,6,7], # [1,2, .., 7] if also want to run  all modes 
                # 'stor_modes': [3, 4, 5], # [1,2, .., 7] if also want to run  all modes 
                'stor_spectroscopy': True,
                'sideband_freq_sweep': True,
                'sideband_length_rabi': True,
                # 'storage_t1': True
                }

# Qubit characterization

## Resonator 

### Resonator Spectroscopy

Fitting parameters are wrong because of using the hanger function (more or less reflection/2), instead of transmission. Is this an easy fix?

In [ ]:
def do_res_spec(
    config_thisrun,
    start=None,
    step=None,
    expts=900,
    reps=500,
    rounds=1,
    pulse_e=False,
    prepulse = {}, # should be a dict 
    gain=0.11,
    length=3.0,
    frequency=7604.3,
    final_delay=250,
    span=10, 
    analyze_and_display=True,
    use_config_params_for_readout = True, # whether to use the readout parameters defined in the config file (True by default) or to override them with the function arguments (False)
):
    """
    prepulse is a dict with key: pulse name and value: dict with keys:
                    "freq": pp.freq,
                    "chan": pp.chan,
                    "sigma": pp.sigma,
                    "n_sigmas": pp.get("n_sigmas", 4), # Default to 4 if missing
                    "gain": pp.gain,
                    "type": pp.get("type", "gauss"),    # Support for 'flat_top', gaussian, 
                    "ramp_sigma": pp.get("ramp_sigma", 0.02), # for flat_top
    """
    rspec = meas.ResonatorSpectroscopyExperiment(
        cfg_dict=cfg_dict, prefix='ResonatorSpectroscopyExperiment',
    )
    if use_config_params_for_readout:
        #change function arguments to match config parameters
        gain = config_thisrun.device.readout.gain
        length = config_thisrun.device.readout.length
        frequency = config_thisrun.device.readout.frequency
        print(f"Using readout parameters from config file: gain={gain}, length={length}, frequency={frequency}")
    rspec.cfg = AttrDict(deepcopy(config_thisrun))

    if start is None:
        start = frequency - span / 2
        print(f"Start frequency not provided. Setting start to {start} MHz based on center frequency and span.")
    if step is None:
        step = span / expts
    
    rspec.cfg.expt = dict(
        start=start,
        step=step,
        expts=expts,
        reps=reps,
        rounds=rounds,
        pulse_e=pulse_e,
        prepulse = prepulse,
        gain=gain,
        length=length,
        frequency=frequency,
        final_delay=final_delay
    )

    rspec.go(analyze=False, display=False, progress=True, save=True)
    if analyze_and_display:
        spec_analysis = meas.SpectroscopyFitting(title = 'Readout Spectroscopy', data=rspec.data)
        spec_analysis.analyze()
        spec_analysis.display()
    return rspec


def update_res_spec(rspec, config_thisrun):
    config_thisrun.device.readout.frequency = [rspec.data['fit'][0]]
    print('Updated readout frequency!')


In [ ]:
config_thisrun.device.readout

In [ ]:
# if expts_to_run['res_spec']:  
# 7387.7, 7493.3, 7524.9, 7588.2?, 7599.5, 7914.2, 7494.3
rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, 
                    frequency = 7600, span = 30, reps = 100, expts = 300, gain=0.2, length = 3, final_delay=250)
spec_analysis = meas.SpectroscopyFitting(data=rspec.data, station = station)
spec_analysis.analyze()
spec_analysis.display(title = 'Readout Spectroscopy')

In [ ]:
config_thisrun.device.readout.frequency = 7600.391

### Sweep resonator gains

In [ ]:
gains = [0.01, 0.03, 0.07, 0.1, 0.2, 0.4, 0.7]
rspecs = []
for i in gains:
    print(f'Running resonator spectroscopy with gain = {i}')
    rspec = do_res_spec(config_thisrun=config_thisrun, pulse_e=False, gain = i, analyze_and_display=False, reps = 300, span = 5, expts = 300)
    rspecs.append(rspec)
# spec_analysis = meas.Spectroscopy(data=rspecs[0].data)
# print("spec_analysis: ", spec_analysis)
# _ = spec_analysis.analyze(data_list = [x.data for x in rspecs])
# spec_analysis.display(data_list = [x.data for x in rspecs])
# print chi 
# chi = man_rspecs[1].data['fit_amps'][2] - man_rspecs[0].data['fit_amps'][2]
# print('Chi from resonator spectroscopy: ', chi, ' MHz')

In [ ]:
for rspec in rspecs:
    spec_analysis = meas.SpectroscopyFitting(data=rspec.data, station = station)
    spec_analysis.analyze()
    spec_analysis.display(title = 'Readout Spectroscopy with gain = ' + str(rspec.cfg.expt.gain))

### Mode frequency tests with tailored reps and len for linewidths

In [ ]:
f_tests = [7493.863, 7525.123]
#+ [0.453, 0.177] from VNA
for freq in f_tests:
    rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, 
                    frequency = freq, span = 7, reps = 400, expts = 100, gain=0.5, final_delay=250)
    spec_analysis = meas.SpectroscopyFitting(data=rspec.data, station = station)
    spec_analysis.analyze()
    spec_analysis.display(title = 'Readout Spectroscopy')

In [ ]:
f_tests = [7588.084]
#-0.096 from VNA    

for freq in f_tests:
    rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, 
                    frequency = freq, span = 2, reps = 500, expts = 500, gain=0.5, length=10.0, final_delay=250)
    spec_analysis = meas.SpectroscopyFitting(data=rspec.data, station = station)
    spec_analysis.analyze()
    spec_analysis.display()

#### Finer sweeps

In [ ]:
f_tests = [7387.69491, 7599.607]

for freq in f_tests:
    rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, 
                    frequency = freq, span = 0.5, reps = 600, expts = 1400, gain=0.7, length=15.0, final_delay=250)
    spec_analysis = meas.SpectroscopyFitting(data=rspec.data, station = station)
    spec_analysis.analyze()
    spec_analysis.display(title = f'Readout Spectroscopy - Gain: 0.7, Cen_Frequency: {freq} MHz')
    max_idx = rspec.data['amps'].argmax()
    print(f"Frequency at max amplitude: {rspec.data['xpts'][max_idx]} MHz")
    print(f"Max amplitude value: {rspec.data['amps'][max_idx]}")

In [ ]:
f_tests = [7387.676, 7599.445]
# f_tests = [7524.9, 7494.3]

for freq in f_tests:
    rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, 
                    frequency = freq, span = 20, reps = 700, expts = 7000, gain=0.8, final_delay=250)
    spec_analysis = meas.SpectroscopyFitting(data=rspec.data, station = station)
    spec_analysis.analyze()
    spec_analysis.display(title= f'Readout Spectroscopy - Cen_Frequency: {freq} MHz')
    max_idx = rspec.data['amps'].argmax()
    print(f"Frequency at max amplitude: {rspec.data['xpts'][max_idx]} MHz")
    print(f"Max amplitude value: {rspec.data['amps'][max_idx]}")

#### bruteforce search possible locations + gains 

In [ ]:
f_tests = [7600]
# f_tests = [7524.9, 7494.3]
ranges = [0.0] #-0.3, -0.2,  , 0.2, 0.3]
g = [0.05, 0.1, 0.15, 0.2]

# g= [0.7]
for gain in g:
    for freq in f_tests:
        for test in ranges:
            rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, 
                            frequency = freq+test, span = 20, reps = 100, expts = 2500, gain=gain, final_delay=250)
            spec_analysis = meas.SpectroscopyFitting(data=rspec.data, station = station)
            spec_analysis.analyze()
            spec_analysis.display(title = f'Readout Spectroscopy - Gain: {gain}, Cen_Frequency: {freq+test} MHz')
            max_idx = rspec.data['amps'].argmax()
            print(f"Frequency at max amplitude: {rspec.data['xpts'][max_idx]} MHz")
            print(f"Max amplitude value: {rspec.data['amps'][max_idx]}")

### Check excited and chis

#### Continuous Qubit Evolution

In [ ]:
pi_gain_fractions = [0.25, 0.5, 0.75, 1.0]
rspecs = []
for fraction in pi_gain_fractions:
    print(f'Running resonator spectroscopy with pi pulse gain fraction = {fraction}')

    rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, pulse_e = False, reps = 300, expts = 500,
                        prepulse = {"custom_pulse": {
            "freq": config_thisrun.device.qubit.f_ge,
            "chan": config_thisrun.hw.soc.dacs.qubit.ch,
            "sigma": config_thisrun.device.qubit.pulses.pi_ge.sigma,
            "length": config_thisrun.device.qubit.pulses.pi_ge.length,
            "gain": fraction * config_thisrun.device.qubit.pulses.pi_ge.gain,
            "type": config_thisrun.device.qubit.pulses.pi_ge.get("type", "gauss"),
            "n_sigmas": config_thisrun.device.qubit.pulses.pi_ge.get("n_sigmas", 4),
            "ramp_sigma": config_thisrun.device.qubit.pulses.pi_ge.get("ramp_sigma", 0.02)}
        })
    rspecs.append(rspec)
# spec_analysis = meas.SpectroscopyFitting(data=rspec.data, station = station)
# spec_analysis.analyze()
# spec_analysis.display()

In [ ]:
# Analyze rspecs at different fractions ... plot them all on the same graph
spec_analysis = meas.SpectroscopyFitting(data=rspecs[0].data, station = station)
_ = spec_analysis.analyze(data_list = [x.data for x in rspecs])
spec_analysis.display(data_list = [x.data for x in rspecs][:4], vlines = [7600.391, 7599.52, 7598.5], hlines = [0.0, 1.5], title = 'Readout Spectroscopy with different prepulse gains', 
                      data_labels = [f'Pi pulse gain fraction: {fraction}' for fraction in pi_gain_fractions][:4], 
                      figsize=(15, 15))

#### Pulse E

In [ ]:
rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, pulse_e = False, reps = 300, expts = 500,
                    prepulse = {"custom_pulse": {
        "freq": config_thisrun.device.qubit.f_ge,
        "chan": config_thisrun.hw.soc.dacs.qubit.ch,
        "sigma": config_thisrun.device.qubit.pulses.pi_ge.sigma,
        "length": config_thisrun.device.qubit.pulses.pi_ge.length,
        "gain": 1.5*config_thisrun.device.qubit.pulses.pi_ge.gain,
        "type": config_thisrun.device.qubit.pulses.pi_ge.get("type", "gauss"),
        "n_sigmas": config_thisrun.device.qubit.pulses.pi_ge.get("n_sigmas", 4),
        "ramp_sigma": config_thisrun.device.qubit.pulses.pi_ge.get("ramp_sigma", 0.02)}
    })
spec_analysis = meas.SpectroscopyFitting(data=rspec.data, station = station)
spec_analysis.analyze()
spec_analysis.display()

In [ ]:
spec_analysis.display(vlines = [7600.391, 7599.52, 7598.5], hlines = [0.0, 1.5])

#### Chi to qubit

In [ ]:
pulse_e_arr = [False, True]
rspecs = []
for pulse_e in pulse_e_arr:
    print(f'Running resonator spectroscopy with pulse_e = {pulse_e}')
    rspec = do_res_spec(config_thisrun=config_thisrun, pulse_e=pulse_e,  analyze_and_display=False, reps = 500, expts = 300)           
    rspecs.append(rspec)
    # update_res_spec(rspec, config_thisrun)
#plot and analyze both
# rspec[0] = meas.Spectroscopy(data=rspecs[0].data)
# rspec[0].analyze()
# rspec[0].display()
# rspec[1] = meas.Spectroscopy(data=rspecs[1].data)
# rspec[1].analyze()
# rspec[1].display()

In [ ]:
spec_analysis = meas.SpectroscopyFitting(data=rspecs[0].data, station = station)
_ = spec_analysis.analyze(data_list = [x.data for x in rspecs])
spec_analysis.display(data_list = [x.data for x in rspecs], vlines = [7600.391, 7599.52, 7598.5])
# print chi 
chi = rspecs[1].data['fit_amps'][2] - rspecs[0].data['fit_amps'][2]
print('Chi from resonator spectroscopy: ', chi, ' MHz')

In [ ]:
# config_thisrun.device.readout.frequency = rspecs[1].data['fit_amps'][2]
# config_thisrun.device.readout.gain = 0.08
# station.handle_config_update(True)

#### Chi to Manipulate 

In [ ]:
prepulse =  {"man_pulse": {"freq": config_thisrun.device.manipulate.f_ge[0],
                            "chan": config_thisrun.hw.soc.dacs.manipulate_in.ch,
                            "sigma": 0,
                            "length": 0.1,
                            "n_sigmas": 0, # Default to 4 if missing
                            "gain": 0.2,
                            "type": "const",    # Support for 'flat_top', gaussian, 
                            "ramp_sigma": 0,}} # for flat_top")

In [ ]:
prepulse

In [ ]:
rspec = do_res_spec(config_thisrun=config_thisrun, pulse_e=False, gain = 0.08, analyze_and_display=False, reps = 300, prepulse=prepulse, span = 5)
spec_analysis = meas.Spectroscopy(data=rspec.data)


In [ ]:
spec_analysis.analyze()
spec_analysis.display()

In [ ]:
rspec.data.keys()

In [ ]:
prepulse =  {"man_pulse": {"freq": config_thisrun.device.manipulate.f_ge[0],
                            "chan": config_thisrun.device.manipulate_in.ch,
                            "sigma": 0,
                            "length": 1,
                            "n_sigmas": 0, # Default to 4 if missing
                            "gain": 0.1,
                            "type": "const",    # Support for 'flat_top', gaussian, 
                            "ramp_sigma": 0,}} # for flat_top")

In [ ]:
# gains = [0.01, 0.03, 0.07, 0.1, 0.2, 0.4, 0.7]
# man_rspecs = []
# for i in gains:
#     prepulse =  {"man_pulse": {"freq": config_thisrun.device.manipulate.f_ge[0],
#                             "chan": config_thisrun.hw.soc.dacs.manipulate_in.ch,
#                             "sigma": 0,
#                             "length": 0.1,
#                             "n_sigmas": 0, # Default to 4 if missing
#                             "gain": i,
#                             "type": "const",    # Support for 'flat_top', gaussian, 
#                             "ramp_sigma": 0,}} # for flat_top")
    
#     print(f'Running resonator spectroscopy with man gain = {i}')
#     rspec = do_res_spec(config_thisrun=config_thisrun, pulse_e=False, gain = 0.08, analyze_and_display=False, reps = 300, prepulse=prepulse, span = 5)
#     man_rspecs.append(rspec)
# spec_analysis = meas.Spectroscopy(data=man_rspecs[0].data)
# print("spec_analysis: ", spec_analysis)
# _ = spec_analysis.analyze(data_list = [x.data for x in man_rspecs])
# spec_analysis.display(data_list = [x.data for x in man_rspecs])
# # print chi 
# # chi = man_rspecs[1].data['fit_amps'][2] - man_rspecs[0].data['fit_amps'][2]
# # print('Chi from resonator spectroscopy: ', chi, ' MHz')

## Qubit Alive Test

In [ ]:
# Run resonator spectroscopy at different logarithmic gains with LIVE 2D COLORPLOT
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

# Define logarithmic gain sweep (from high to low, top to bottom)
gains = np.logspace(0, -4, 100)[::-1]  # 20 gains from 1.0 down to 0.001 (reversed)
gains = gains[::-1]  # Start from highest gain (1.0) and go down
print(f"Testing {len(gains)} gains (high to low): {gains}")

# Store all results
rspecs = []
peak_amps = []
peak_freqs = []

# Initialize 2D array to store amplitude data (gains x frequencies)
amp_matrix = None
freq_array = None

# Setup live plotting
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
plt.ion()  # Turn on interactive mode

# Run res spec for each gain with live updates
for i, gain in enumerate(gains):
    # Calculate reps: 5x more reps for each factor of 10 decrease in gain
    # At gain=1.0, reps=100; at gain=0.1, reps=500; at gain=0.01, reps=2500
    # Cap reps at 1000 maximum
    reps = int(100 * (1.0 / gain) ** 0.69897)  # 0.69897 = log10(5)
    reps = min(reps, 5000)  # Floor at 5000 reps maximum
    
    print(f"\nRunning res spec {i+1}/{len(gains)} with gain = {gain:.4f}, reps = {reps}")
    
    rspec = do_res_spec(
        config_thisrun=config_thisrun, 
        analyze_and_display=False,
        frequency=7590,  # Center frequency
        span=30,        # Span around center
        reps=reps,       # Dynamic reps based on gain
        expts=300,       # Number of frequency points
        gain=gain        # Logarithmic gain value
    )
    
    rspecs.append(rspec)
    
    # Extract data for this run
    xpts = rspec.data['xpts']
    avgi = rspec.data['avgi']
    avgq = rspec.data['avgq']
    
    # Handle different data structures (could be nested or flat)
    if isinstance(avgi, (list, np.ndarray)) and len(np.array(avgi).shape) > 1:
        avgi = np.array(avgi)[0]
        avgq = np.array(avgq)[0]
    
    amps = np.abs(np.array(avgi) + 1j*np.array(avgq))
    peak_idx = np.argmax(amps)
    peak_amps.append(float(amps[peak_idx]))
    peak_freqs.append(float(xpts[peak_idx]))
    
    # Initialize or update amplitude matrix
    if amp_matrix is None:
        freq_array = xpts
        amp_matrix = np.zeros((len(gains), len(xpts)))
    amp_matrix[i, :] = amps
    
    # Update plots
    axes[0].clear()
    axes[1].clear()
    
    # Plot 1: 2D colorplot (heatmap) - Gain vs Frequency
    completed_rows = i + 1
    im = axes[0].pcolormesh(freq_array, gains[:completed_rows], amp_matrix[:completed_rows, :], 
                            shading='auto', cmap='viridis')
    axes[0].set_yscale('log')
    axes[0].set_xlabel('Frequency (MHz)', fontsize=12)
    axes[0].set_ylabel('Readout Gain', fontsize=12)
    axes[0].set_title(f'Resonator Spectroscopy: Gain vs Frequency (Completed: {completed_rows}/{len(gains)})', 
                      fontsize=13, fontweight='bold')
    
    # Add colorbar
    if not hasattr(axes[0], '_colorbar'):
        axes[0]._colorbar = plt.colorbar(im, ax=axes[0], label='Amplitude (ADC units)')
    else:
        axes[0]._colorbar.update_normal(im)
    
    # Plot 2: Peak amplitude vs gain
    axes[1].loglog(gains[:len(peak_amps)], peak_amps, 'o-', linewidth=2, markersize=8, color='C1')
    axes[1].set_xlabel('Readout Gain (log scale)', fontsize=12)
    axes[1].set_ylabel('Peak Amplitude (ADC units, log scale)', fontsize=12)
    axes[1].set_title('Peak Amplitude vs Readout Gain', fontsize=13, fontweight='bold')
    axes[1].grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    clear_output(wait=True)
    display(fig)
    
plt.ioff()  # Turn off interactive mode
print(f"\nCompleted all {len(gains)} gain sweeps!")
print(f"Peak frequencies: {peak_freqs}")
print(f"Peak amplitudes: {peak_amps}")

In [ ]:
# Plot all resonator spectroscopy results at different gains
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Plot 1: All spectra overlaid
ax1 = axes[0]
for i, (rspec, gain) in enumerate(zip(rspecs, gains)):
    xpts = rspec.data['xpts']
    avgi = rspec.data['avgi'][0]
    avgq = rspec.data['avgq'][0]
    amps = np.abs(avgi + 1j*avgq)
    ax1.plot(xpts, amps, label=f'Gain={gain:.4f}', alpha=0.7)

ax1.set_xlabel('Frequency (MHz)')
ax1.set_ylabel('Amplitude (ADC units)')
ax1.set_title('Resonator Spectroscopy at Different Gains')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(True, alpha=0.3)

# Plot 2: Peak amplitude vs gain (log scale)
ax2 = axes[1]
peak_amps = []
peak_freqs = []

for rspec in rspecs:
    xpts = rspec.data['xpts']
    avgi = rspec.data['avgi'][0]
    avgq = rspec.data['avgq'][0]
    amps = np.abs(avgi + 1j*avgq)
    peak_idx = np.argmax(amps)
    peak_amps.append(amps[peak_idx])
    peak_freqs.append(xpts[peak_idx])

ax2.semilogx(gains, peak_amps, 'o-', linewidth=2, markersize=8)
ax2.set_xlabel('Readout Gain (log scale)')
ax2.set_ylabel('Peak Amplitude (ADC units)')
ax2.set_title('Peak Amplitude vs Readout Gain')
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f"\nPeak frequencies found: {peak_freqs}")
print(f"Peak amplitudes: {peak_amps}")

In [ ]:
qspec =  do_pulse_probe_spec(
    config_thisrun,
    center= 2061, #3063, #config_thisrun.device.qubit.f_ge - 1,
    span=50,
    expts=80, #300,
    reps=500,
    rounds=1,
    prepulse={"pi_qubit_ge":{}, "pi_qubit_ef":{}},  # should be a dict
    
    probe_pulse_param = {
    "gain": 0.5,
    "length": 1.0, #0.03?
    "freq": None,  # doesn't matter
    "chan": config_thisrun.hw.soc.dacs.qubit.ch,
    "sigma": 0,
    "sigma_inc": 0,
    "ramp_sigma": None,
    "ramp_sigma_inc": None,
    "phase":0,
    "type": 'const',},
    analyze_and_display = False,
    sweep_other_param = sweep_other_param
)

In [ ]:
3492.43 + 3369.277 - 4800

In [ ]:
xlist = qspec.data['xpts_freq']
ylist = qspec.data['xpts_gain']
# ylist = qspec.data['xpts_length']
zlists = [qspec.data['avgi'], qspec.data['avgq']]
# print('zlist shape: ', np.array(zlists).shape)
# print(zlists)
zlists = [z.T for z in zlists]
# print('zlist shape after transpose: ', np.array(zlists).shape)
# print(zlists)
zlists_adjusted = [z[:-1, :-1] for z in zlists]  # Adjust dimensions for shading='flat'
freq_guess = config_thisrun.device.qubit.f_ge
print(zlists_adjusted)
plot = meas.ColorPlot2D(xlist = xlist, ylist = ylist,
                        zlists = zlists_adjusted,
                        xlabel = 'freq', ylabel = 'gain ')
              #    xlabel = 'freq', ylabel = 'len') #, title = 'T2 Ramsey Scan')
plot.display() #vlines=[], hlines = [0.5, 1.5, 2, 2.5])

## Single Shot

In [ ]:
def do_single_shot(
    config_thisrun,
    readout_gain=None, 
    readout_length=None,
    readout_frequency=None,
    shots=5000,
    pulse_e = True, 
    prepulse = {}, 
    sweep_params = {},
    final_delay = {},
    analyze_and_display = True,
   
):
    """
    Run the single shot experiment with configurable parameters.
    """

    hstgrm = meas.SingleShotExperiment(
        cfg_dict=cfg_dict, prefix='SingleShot',
    )

    hstgrm.cfg = AttrDict(deepcopy(config_thisrun))
    if readout_gain is None:
        readout_gain = station.yaml_cfg.device.readout.gain
    if readout_length is None:
        readout_length = station.yaml_cfg.device.readout.length
    if readout_frequency is None:
        readout_frequency = station.yaml_cfg.device.readout.frequency
    print('Using readout parameters - Gain: {}, Length: {}, Frequency: {}'.format(readout_gain, readout_length, readout_frequency))

    hstgrm.cfg.expt = {
    "shots": shots,  # Number of shots per experiment
    "reps": 1,  # Number of repetitions
    "rounds": 1,  # Number of software averages
    "pulse_e": pulse_e,
    "prepulse": prepulse,
    "sweep_param": sweep_params,
    "final_delay": final_delay,
    "readout_gain": readout_gain,
    "readout_length": readout_length,
    "readout_frequency": readout_frequency,
    }

    hstgrm.go(analyze=False, display=False, progress=True, save=True)
    if analyze_and_display:
        hst_analysis = meas.Histogram(hstgrm.data, station = station )
        _ = hst_analysis.analyze()
        hst_analysis.display()
        return hst_analysis
        
    return hstgrm


def singleshot_postproc(station, expt):
    # expt.analyze(plot = True, station=station, subdir=station.autocalib_path)
    fids = float(expt.results['fidelity'])
    # confusion_matrix = expt.results['confusion_matrix']
    thresholds_new = float(expt.results['threshold'])
    angle = float(expt.results['rotation_angle'])
    print(fids)

    config_thisrun = station.config_thisrun
    config_thisrun.device.readout.phase = config_thisrun.device.readout.phase + angle
    config_thisrun.device.readout.threshold = thresholds_new
    # config_thisrun.device.readout.threshold_list = [thresholds_new]
    # config_thisrun.device.readout.Ie = [np.median(expt.data['Ie_rot'])]
    # config_thisrun.device.readout.Ig = [np.median(expt.data['Ig_rot'])]
    # if expt.cfg.expt.active_reset:
    #     config_thisrun.device.readout.confusion_matrix_with_active_reset = confusion_matrix
    # else:
    #     config_thisrun.device.readout.confusion_matrix_without_reset = confusion_matrix
    print('Updated readout!')



In [ ]:
# 'max_x': np.float64(0.15830433986449366),
#   'max_y': np.float64(7605.093303108215),

hstgrm = do_single_shot(config_thisrun, 
                            shots = 10000, final_delay = 250, analyze_and_display = True,
                            # readout_frequency = 7605.093303108215, 
                            readout_gain = 0.3,
                            readout_length = np.int64(10),
                            sweep_params={})

In [ ]:
# singleshot_postproc(station, hstgrm)
# station.handle_config_update(True)

In [ ]:
config_thisrun.device.readout.phase

In [ ]:
config_thisrun.device.readout

In [ ]:
hst_analysis.results

### Single Shot Sweep

In [ ]:
# Define experimental parameters
freq_span = 3   # Mhz
freq_start = config_thisrun.device.readout.frequency -freq_span/2 
freq_expts = 30

# length_start = 1 # us
# length_span = 20  # us
# length_expts = 10

gain_start = 0.05 # dB
gain_span = 0.15   # dB
gain_expts = 30

span = 0.1  # General span for other parameters

# Define sweep_other_param dictionary
sweep_other_param = {
    "readout_frequency": {
        "label": "readout_pulse",
        "param": "freq",
        "param_type": "pulse",
        "expts": freq_expts,
        "start": freq_start,
        "step": freq_span/ freq_expts
    },
    # "readout_length": {
    #     "label": "readout_pulse",
    #     "param": "length",
    #     "param_type": "pulse",
    #     "expts": length_expts,
    #     "start": length_start,
    #     "step": length_span/length_expts
    # },
    "readout_gain": {
        "label": "readout_pulse",
        "param": "gain",
        "param_type": "pulse",
        "expts": gain_expts,
        "start": gain_start,
        "step": gain_span/gain_expts
    }
}

In [ ]:
hstgrm = do_single_shot(config_thisrun, 
                            shots = 500, final_delay = 250, analyze_and_display = False,
                            readout_length = 5,
                            sweep_params=sweep_other_param)

In [ ]:
hst_analysis = meas.Histogram(hstgrm.data, station = station, sweep_params=sweep_other_param)
_ = hst_analysis.analyze_swept_histogram_data()
hst_analysis.display_swept_histogram_data()


#### Length Sweep 


In [ ]:
hstgrms = []
lengths = np.arange(3, 20, 2)
for length in lengths: 
    hstgrm = do_single_shot(config_thisrun, 
                            shots = 5000, final_delay = 250, analyze_and_display = False,
                            readout_length = length,
                            sweep_params=sweep_other_param)
    hstgrms.append(hstgrm)
    

In [ ]:
results = []
for hst in hstgrms:
    hst_analysis = meas.Histogram(hst.data, station = station, sweep_params=sweep_other_param)
    _ = hst_analysis.analyze_swept_histogram_data()
    hst_plot_object = hst_analysis.display_swept_histogram_data()
    results.append(hst_plot_object.results)

In [ ]:
results[-3]

In [ ]:
hstgrms[-3].cfg

In [ ]:
# result in results have the format 
# [{'z_index': 0,
#   'max_value': np.float64(0.5257597573206373),
#   'max_x': np.float64(0.08093755722395166),
#   'max_y': np.float64(7602.7143081665035),
#   'x_idx': np.int64(6),
#   'y_idx': np.int64(2)}]
# make 3 plots all with x axis being length, y1 = max fidelity, y2 = max gain, y3 = max frequency
def _get(field, item):
    if isinstance(item, (list, tuple)) and len(item) > 0 and isinstance(item[0], dict):
        d = item[0]
    elif isinstance(item, dict):
        d = item
    else:
        d = {}
    return float(d.get(field, np.nan))

x = np.array(lengths)

y_fid = np.array([_get('max_value', r) for r in results])
y_gain = np.array([_get('max_x', r) for r in results])
y_freq = np.array([_get('max_y', r) for r in results])

fig, axes = plt.subplots(3, 1, figsize=(7, 9), sharex=True)
axes[0].plot(x, y_fid, '-o')
axes[0].set_ylabel('Max fidelity')
axes[0].grid(True)

axes[1].plot(x, y_gain, '-o')
axes[1].set_ylabel('Max gain')
axes[1].grid(True)

axes[2].plot(x, y_freq, '-o')
axes[2].set_ylabel('Max frequency')
axes[2].set_xlabel('Length')
axes[2].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
hstgrms[0].data

In [ ]:
hist_tup = ()

for hist in range(len(hstgrms)):
    hist_tup = hist_tup + (hstgrms[hist].data,)
print(hist_tup)

hst_analysis = meas.Histogram(hist_tup, station = station )
parsed_hist = hst_analysis.parse_swept_histogram_data(sweep_params = sweep_other_param)
# hist_analysis.display()
print(parsed_hist.keys())
    

In [ ]:
hst_analysis = meas.Histogram(hstgrms.data, station = station )
parsed_hist = hst_analysis.parse_swept_histogram_data(sweep_params = sweep_other_param)

# find fidelity 
hst_analysis.analyze_swept_histogram_data(parsed_hist)
parsed_hist.keys()

In [ ]:
parsed_hist['sweep_keys_and_xpts']

In [ ]:
print(np.shape(parsed_hist['Ig']))
print(np.shape(parsed_hist['sweep_keys_and_xpts'][keys[0]]))
print(np.shape(parsed_hist['sweep_keys_and_xpts'][keys[1]]))

In [ ]:

# plot.display()

In [ ]:
plot_object.results

In [ ]:
# config_thisrun.device.readout.frequency = float(plot_object.results[0]['max_y'])
# config_thisrun.device.readout.gain = float(plot_object.results[0]['max_x'])

In [ ]:
# station.handle_config_update(True)

In [ ]:
keys = ['Ig', 'Ie', 'Qg', 'Qe']
for key in keys: 
    print(np.mean(hstgrm.data[key]))

In [ ]:
hst_analysis.display_old()

In [ ]:
hstgrm.data

In [ ]:
dir(meas)

In [ ]:
hstgrm[1].analyze(plot = True)

In [ ]:
# if expts_to_run['single_shot']:
#     update_single_shot(hstgrm[1], config_thisrun)
#     print('Single shot done!')

In [ ]:
# hstgrm.cfg

## JPA Calibration

In [ ]:
def do_jpa_current_sweep( config_thisrun,
    expt_path,
    config_path,
    jpa_current_start=-8,
    jpa_current_step=0.1,
    jpa_current_stop = -2,
    qubits=[0],
    reps=5000,
    check_f=False,
    active_reset=False,
    man_reset=False,
    storage_reset=False,
    qubit=0,
    pulse_manipulate=False,
    cavity_freq=4984.373226159381,
    cavity_gain=800,
    cavity_length=2,
    prepulse=False,
    pre_sweep_pulse=None,
    gate_based=True,
    relax_delay=2500
):
    """Run the single shot experiment with configurable parameters."""
    from multimode_expts.sequential_experiment_classes import histogram_sweep_class
    experiment_class = histogram_sweep_class
    sweep_experiment_name = 'histogram_jpa_current_sweep'
    class_for_exp = experiment_class(soccfg=soc, path=expt_path, prefix=sweep_experiment_name, config_file=config_path, exp_param_file=exp_param_file)

    class_for_exp.yaml_cfg = AttrDict(deepcopy(config_thisrun))
    
    class_for_exp.loaded[sweep_experiment_name] =  {
        'qubits': qubits,
        'reps': reps,
        'check_f': check_f,
        'active_reset': active_reset,
        'man_reset': man_reset,
        'storage_reset': storage_reset,
        'qubit': qubit,
        'pulse_manipulate': pulse_manipulate,
        'cavity_freq': cavity_freq,
        'cavity_gain': cavity_gain,
        'cavity_length': cavity_length,
        'prepulse': prepulse,
        'pre_sweep_pulse': pre_sweep_pulse,
        'gate_based': gate_based,
        'jpa_current_start': jpa_current_start,
        'jpa_current_step': jpa_current_step,
        'jpa_current_stop': jpa_current_stop,
        'relax_delay': relax_delay
    }

   
    class_for_exp.yaml_cfg.device.readout.relax_delay = [relax_delay]  # Wait time between experiments [us]

    return eval('class_for_exp.run_sweep')( sweep_experiment_name = sweep_experiment_name)

In [ ]:
# sweep_func = do_jpa_current_sweep(config_thisrun, expt_path,
#                                    config_path, reps=1000,
#                                     #  jpa_current_start=0., jpa_current_step=0.2,
#                                     #    jpa_current_stop=10., 
#                                       #  active_reset=True,
#                                          relax_delay=5)

In [ ]:
from slab.instruments import YokogawaGS200
dcflux = YokogawaGS200(address="192.168.137.149")
dcflux.set_output(True)
dcflux.set_mode('current')

jpa_current = -1.875 # mA
jpa_current = 0 # mA
current = jpa_current * 1e-3  # Convert from mA to A
dcflux.set_current(current)

print(1e3 * dcflux.get_current(), 'mA')

#### Sweep both gain and current

In [ ]:
def do_jpa_current_gain_sweep( config_thisrun,
    expt_path,
    config_path,
    jpa_current_start=3.7,
    jpa_current_step=0.005,
    jpa_current_stop = 3.8,
    jpa_gain_start=-15,
    jpa_gain_step=0.5,
    jpa_gain_stop = -11,
    qubits=[0],
    reps=1000,
    check_f=False,
    active_reset=False,
    man_reset=False,
    storage_reset=False,
    qubit=0,
    pulse_manipulate=False,
    cavity_freq=4984.373226159381,
    cavity_gain=800,
    cavity_length=2,
    prepulse=False,
    pre_sweep_pulse=None,
    gate_based=True,
    relax_delay=2500
):
    """Run the single shot experiment with configurable parameters."""
    from multimode_expts.sequential_experiment_classes import histogram_sweep_class
    experiment_class = histogram_sweep_class
    sweep_experiment_name = 'histogram_jpa_gain_current_sweep'
    class_for_exp = experiment_class(soccfg=soc, path=expt_path, prefix=sweep_experiment_name, config_file=config_path, exp_param_file=exp_param_file)

    class_for_exp.yaml_cfg = AttrDict(deepcopy(config_thisrun))
    
    class_for_exp.loaded[sweep_experiment_name] =  {
        'qubits': qubits,
        'reps': reps,
        'check_f': check_f,
        'active_reset': active_reset,
        'man_reset': man_reset,
        'storage_reset': storage_reset,
        'qubit': qubit,
        'pulse_manipulate': pulse_manipulate,
        'cavity_freq': cavity_freq,
        'cavity_gain': cavity_gain,
        'cavity_length': cavity_length,
        'prepulse': prepulse,
        'pre_sweep_pulse': pre_sweep_pulse,
        'gate_based': gate_based,
        'jpa_current_start': jpa_current_start,
        'jpa_current_step': jpa_current_step,
        'jpa_current_stop': jpa_current_stop,
        'jpa_gain_start': jpa_gain_start,
        'jpa_gain_step': jpa_gain_step,
        'jpa_gain_stop': jpa_gain_stop,
    }

    print('relax delay set to:', relax_delay)
    class_for_exp.yaml_cfg.device.readout.relax_delay = [relax_delay]  # Wait time between experiments [us]

    return eval('class_for_exp.run_sweep')( sweep_experiment_name = sweep_experiment_name)

In [ ]:
# sweep_func = do_jpa_current_gain_sweep(config_thisrun, expt_path,
#                                         config_path, reps=1000, 
#                                         jpa_current_start=0,
#                                         jpa_current_step=0.05,
#                                         jpa_current_stop=5,
#                                         jpa_gain_start=-15,
#                                         jpa_gain_step=0.5,
#                                         jpa_gain_stop=-11,
#                                         # active_reset=True, relax_delay=5
#                                         )

## Qubit ge

### Pulse-probe

In [ ]:
def do_pulse_probe_spec(
    config_thisrun,
    start=None,
    center = None,
    step=None,
    span = None,
    expts=300,
    reps=100,
    rounds=1,
    pulse_e=False,
    prepulse={},  # should be a dict
    sweep_other_param=None,  # Additional sweep parameters
    probe_pulse_param = {},
    # readout_probe_gain = None, # bad code
    analyze_and_display=True,
):
    """
    Perform a pulse probe spectroscopy experiment.

    Parameters:
        config_thisrun (dict): Configuration dictionary.
        start (float, optional): Start frequency.
        step (float, optional): Frequency step size.
        expts (int, optional): Number of experiments.
        reps (int, optional): Number of repetitions per experiment.
        rounds (int, optional): Number of rounds.
        pulse_e (bool, optional): Whether to enable pulse_e.
        prepulse (dict, optional): Pre-pulse configuration.
        sweep_other_param (dict, optional): Additional sweep parameters.
        analyze_and_display (bool, optional): Whether to analyze and display results.

    The main pulse parameters should be defined in `self.cfg.expt.probe_pulse_param` as a dictionary with the following keys:
        - "chan": Channel for the pulse
        - "freq": Frequency of the pulse
        - "gain": Gain of the pulse
        - "phase": Phase of the pulse
        - "length": Length of the pulse
        - "type": Type of the pulse (e.g., 'gauss', 'flat_top')
        - "sigma": Sigma value for Gaussian pulses
        - "sigma_inc": Increment for sigma
        - "ramp_sigma": Ramp sigma for flat-top pulses
        - "ramp_sigma_inc": Increment for ramp sigma
    """
    ppspec = meas.PulseProbeSpectroscopyExperiment(
        cfg_dict=cfg_dict, prefix='PulseProbeSpectroscopyExperiment',
    )

    ppspec.cfg = AttrDict(deepcopy(config_thisrun))

    if start is None:
        start = center - span / 2
    if step is None:
        step = span / expts
    
    # print(ppspec.cfg)
    ppspec.cfg.expt = {
            "start": start,
            "step": step,
            "expts": expts,
            "reps": reps,
            "rounds": rounds,
            "pulse_e": pulse_e,
            "prepulse": prepulse,
            "sweep_other_param": sweep_other_param,
            "probe_pulse_param": probe_pulse_param,
            # "readout_probe_gain": readout_probe_gain
            # "sweep_other_param": sweep_other_param
        }
    
    # print(ppspec.cfg.expt)

    ppspec.go(analyze=False, display=False, progress=True, save=True)
    if analyze_and_display:
        spec_analysis = meas.SpectroscopyFitting(data=ppspec.data)
        spec_analysis.analyze()
        spec_analysis.display()
    return ppspec

def update_pulse_probe_ge(qspec, config_thisrun):
    config_thisrun.device.qubit.f_ge = [qspec.data['fit_avgi'][2]]
    print('Updated qubit frequency!')

In [ ]:
qspec =  do_pulse_probe_spec(
    config_thisrun,
    center= 3705, #config_thisrun.device.qubit.f_ge - 123,
    span=20,
    expts=500,
    reps=300,
    rounds=1,
    prepulse= {}, #"pi_qubit_ge":{}},  # should be a dict
    
    probe_pulse_param = {
    "gain": 0.05, #config_thisrun.device.qubit.pulses.pi_ge.gain, #0.123,
    "length": 0.8, #config_thisrun.device.qubit.pulses.pi_ge.sigma *4, #0.2
    "freq": None,  # doesn't matter
    "chan": config_thisrun.hw.soc.dacs.qubit.ch,
    "sigma": 0,
    "sigma_inc": 0,
    "ramp_sigma": None,
    "ramp_sigma_inc": None,
    "phase":0,
    "type": 'const',},
    analyze_and_display = False
)

In [ ]:
spec_analysis = meas.SpectroscopyFitting(station = station, data=qspec.data)
spec_analysis.analyze() #fitparams = [-0.7, -0.5, 3064.7, 10]) #[y0, yscale, x0, xscale]
spec_analysis.display()

In [ ]:
spec_analysis = meas.SpectroscopyFitting(station = station, data=qspec.data)
# spec_analysis.analyze_photon_resolved(fitparams = {'amps': [2.8, -1*1, 3492.4, -1.0, 0.1, 0.6], 
                                                    # 'avgi': [-2.6, 1, 3492.4, -1.0, 0.1, 0.6]})
spec_analysis.display_photon_resolved()

In [ ]:
# [offset, -1*scale, mu, chi, sigma, nbar]
spec_analysis.analyze_photon_resolved()
spec_analysis.display_photon_resolved(fit=True) #fitparams = {'amps': [2.8, -1*1, 3492.4, -1.0, 0.1, 0.6], 
                                                    # 'avgi': [-2.6, 1, 3492.4, -1.0, 0.1, 0.6]})

In [ ]:
config_thisrun.device.readout.final_delay

In [ ]:
spec_analysis = meas.SpectroscopyFitting(station = station, data=qspec.data)
spec_analysis.analyze()
spec_analysis.display()

In [ ]:
if expts_to_run['pulse_probe_ge']:
    update_pulse_probe_ge(qspec, config_thisrun)
    print('Pulse probe spectroscopy done!')

#### Readout temp monitoring

In [ ]:
qspec =  do_pulse_probe_spec(
    config_thisrun,
    center= config_thisrun.device.qubit.f_ge - 2,
    span=5,
    expts=100,
    reps=400,
    rounds=1,
    prepulse= {}, # {"pi_qubit_ge":{}},  # should be a dict
    
    probe_pulse_param = {
    "gain": config_thisrun.device.qubit.pulses.slow_pi_ge.gain, #0.123,
    "length": config_thisrun.device.qubit.pulses.slow_pi_ge.length, #0.2
    "freq": None,  # doesn't matter
    "chan": config_thisrun.hw.soc.dacs.qubit.ch,
    "sigma": 0,
    "sigma_inc": 0,
    "ramp_sigma": None,
    "ramp_sigma_inc": None,
    "phase":0,
    "type": 'const',},
    analyze_and_display = False
)

In [ ]:
# [offset, -1*scale, mu, chi, sigma, nbar]
spec_analysis = meas.SpectroscopyFitting(station = station, data=qspec.data)
spec_analysis.analyze_photon_resolved()
spec_analysis.display_photon_resolved(fit=True) #fitparams = {'amps': [2.8, -1*1, 3492.4, -1.0, 0.1, 0.6], 
                                                    # 'avgi': [-2.6, 1, 3492.4, -1.0, 0.1, 0.6]})

#### Long Overnight 2D sweep to find qubit

In [ ]:
import os
import time
from datetime import datetime

# qfreq = [3820.9, 4421.7]
# for f in qfreq:

#SET SPEED OF EXPERIMENT
# spd = 0.1 #ultra quick 3 min
# spd = 0.3 # 10 min check
spd = 1 # 5hr scan
# spd = 3 #thorough

gain_expts = int(550*spd) #200
gain_start = 0.005
gain_span = 0.95
sweep_other_param = {
    "gain": {
        "label": "probe_pulse",
        "param": "gain",
        "param_type": "pulse",
        "expts": gain_expts,
        "start": gain_start,
        "step": gain_span/ gain_expts,
        "parent_dict": "probe_pulse_param"
        # "parent_dict": "probe_pulse_param"
        },
    }

qspec =  do_pulse_probe_spec(
    config_thisrun,
    center= config_thisrun.device.qubit.f_ge,
    span= int(650*spd),
    expts=int(550*spd),
    reps= int(380*spd),
    rounds=1,
    # readout_probe_gain = 0, 
    prepulse= {}, # {"pi_qubit_ge":{}, "pi_qubit_ef":{}} ,
                #{"readout_probe": {"gain": 0.003,
                    # "length": 3.0,
                    # "freq": config_thisrun.device.readout.frequency,
                    # "chan": config_thisrun.hw.soc.dacs.readout.ch,
                    # "sigma": 0,
                    # "sigma_inc": 0,
                    # "ramp_sigma": None,
                    # "ramp_sigma_inc": None,
                    # "phase":0,
                    # "type": 'const',
                    # "t": 0.084, 
                    # "delay_flag": False}},
    
    probe_pulse_param = {
    "gain": 0.05, #config_thisrun.device.qubit.pulses.pi_ge.gain, #0.123,
    "length": 0.8, #config_thisrun.device.qubit.pulses.pi_ge.sigma *4, #0.2
    "freq": None,  # doesn't matter
    "chan": config_thisrun.hw.soc.dacs.qubit.ch,
    "sigma": 0,
    "sigma_inc": 0,
    "ramp_sigma": None,
    "ramp_sigma_inc": None,
    "phase":0,
    "type": 'const',},
    sweep_other_param = sweep_other_param,
    analyze_and_display = False
)

expt = qspec
xlist = expt.data['xpts_freq']
ylist = expt.data['xpts_gain']
# ylist = expt.data['xpts_length']
zlists = [expt.data['amps'], expt.data['avgi'], expt.data['avgq']]
# print('zlist shape: ', np.array(zlists).shape)
# print(zlists)
zlists = [z.T for z in zlists]
# print('zlist shape after transpose: ', np.array(zlists).shape)
# print(zlists)
zlists_adjusted = [z[:-1, :-1] for z in zlists]  # Adjust dimensions for shading='flat'
freq_guess = config_thisrun.device.qubit.f_ge
# print(zlists_adjusted)
# plot = meas.ColorPlot2D(xlist = xlist, ylist = ylist,
#                         zlists = zlists_adjusted,
#                         xlabel = 'freq', ylabel = 'gain ')
#             #    xlabel = 'freq', ylabel = 'len') #, title = 'T2 Ramsey Scan')
# plot.display() #vlines=[3064.2], hlines = [0.96]

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 12))
c1 = axes[0].pcolor(xlist, ylist, zlists_adjusted[0])
c2 = axes[1].pcolor(xlist, ylist, zlists_adjusted[1])
c3 = axes[2].pcolor(xlist, ylist, zlists_adjusted[2])
axes[0].set_title(f"Colormap of qubit spec vs Frequency (center: {config_thisrun.device.qubit.f_ge} MHz) and Gain")
axes[2].set_xlabel("Frequency (MHz)")

clabels = [c1, c2, c3]
ylabels = ["Amplitude [ADC]", "I [ADC]", "Q [ADC]"]

for i in range(3):
    axes[i].set_ylabel(ylabels[i])
    fig.colorbar(clabels[i], ax=axes[i])
plt.savefig(f"{datetime.now()}_res_spec_vs_current_{config_thisrun.device.qubit.f_ge}MHz.png".replace(' ', '_').replace(':', ''))
print(f'plot saved at: {os.getcwd()}')
plt.show()

#### 2D Qubit Spec for readout photon calibration 

In [ ]:
expts = 40 #200
start = 1e-3
span = 0.02
sweep_other_param = {
    "readout_probe_gain": {
        "label": "readout_probe",
        "param": "gain",
        "param_type": "pulse",
        "expts": expts,
        "start": start,
        "step": span/ expts,
        # "parent_dict": "
        #"parent_dict": "probe_pulse_param"
        
    },
    }

In [ ]:
len_expts = 180 #200
len_start = 0.005
len_span = 4
sweep_other_param = {
    "length": {
        "label": "probe_pulse",
        "param": "length",
        "param_type": "pulse",
        "expts": len_expts,
        "start": len_start,
        "step": len_span/ len_expts,
        "parent_dict": "probe_pulse_param"
        #"parent_dict": "probe_pulse_param"
        
    },
    }

In [ ]:
gain_expts = 120 #200
gain_start = 0.005
gain_span = 0.2
sweep_other_param = {
    "gain": {
        "label": "probe_pulse",
        "param": "gain",
        "param_type": "pulse",
        "expts": gain_expts,
        "start": gain_start,
        "step": gain_span/ gain_expts,
        "parent_dict": "probe_pulse_param"
        #"parent_dict": "probe_pulse_param"
        
    },
    }

In [ ]:
qspec =  do_pulse_probe_spec(
    config_thisrun,
    center= config_thisrun.device.qubit.f_ge - 3,
    span=10,
    expts=60,
    reps=100,
    rounds=1,
    # readout_probe_gain = 0, 
    prepulse= {}, # {"pi_qubit_ge":{}, "pi_qubit_ef":{}} ,
                #{"readout_probe": {"gain": 0.003,
                    # "length": 3.0,
                    # "freq": config_thisrun.device.readout.frequency,
                    # "chan": config_thisrun.hw.soc.dacs.readout.ch,
                    # "sigma": 0,
                    # "sigma_inc": 0,
                    # "ramp_sigma": None,
                    # "ramp_sigma_inc": None,
                    # "phase":0,
                    # "type": 'const',
                    # "t": 0.084, 
                    # "delay_flag": False}},
    
    probe_pulse_param = {
    "gain": 0.22,
    "length": 0.28,
    "freq": None,  # doesn't matter
    "chan": config_thisrun.hw.soc.dacs.qubit.ch,
    "sigma": 0,
    "sigma_inc": 0,
    "ramp_sigma": None,
    "ramp_sigma_inc": None,
    "phase":0,
    "type": 'const',},
    sweep_other_param = sweep_other_param,
    analyze_and_display = False
)

In [ ]:
qspec.data.keys()

In [ ]:
gain1 = 10**(-60 /10) # 0.003
gain2 = 10**(-40/10) # 0.01
n1 = 0.540
n2 = 0.794 
n_per_gain = (n2 - n1) / (gain2 - gain1)
print('n_per_gain_linear: ', n_per_gain, ' photons per unit gain_linear')

In [ ]:
expt = qspec
xlist = expt.data['xpts_freq']
ylist = expt.data['xpts_gain']
# ylist = expt.data['xpts_length']
zlists = [expt.data['avgi'], expt.data['avgq']]
# print('zlist shape: ', np.array(zlists).shape)
# print(zlists)
zlists = [z.T for z in zlists]
# print('zlist shape after transpose: ', np.array(zlists).shape)
# print(zlists)
zlists_adjusted = [z[:-1, :-1] for z in zlists]  # Adjust dimensions for shading='flat'
freq_guess = config_thisrun.device.qubit.f_ge
# print(zlists_adjusted)
plot = meas.ColorPlot2D(xlist = xlist, ylist = ylist,
                        zlists = zlists_adjusted,
                        xlabel = 'freq', ylabel = 'gain ')
              #    xlabel = 'freq', ylabel = 'len') #, title = 'T2 Ramsey Scan')
plot.display() #vlines=[3064.2], hlines = [0.96]

In [ ]:
spec_analysis = meas.SpectroscopyFitting(station = station, data=qspec.data)
# spec_analysis.analyze() #fitparams = [-0.7, -0.5, 3064.7, 10]) #[y0, yscale, x0, xscale]
spec_analysis.display()

In [ ]:
spec_analysis = meas.SpectroscopyFitting(station = station, data=qspec.data)
spec_analysis.analyze_photon_resolved(fitparams = {'amps': [2.8, -1*1, 3492.4, -1.0, 0.1, 0.6], 
                                                    'avgi': [-2.6, 1, 3492.4, -1.0, 0.1, 0.6]})
spec_analysis.display_photon_resolved()

In [ ]:
data[0]

In [ ]:
data[0]['amps'].shape

In [ ]:
fname = '00127_PulseProbeSpectroscopyExperiment.h5'
data = station.load_data(fname)
spec_analysis = meas.SpectroscopyFitting(station = station, data=data[0])
spec_analysis.analyze_photon_resolved_gain_sweep()
spec_analysis.display_photon_resolved_gain_sweep()

#### Qubit Spec Power Sweep

In [ ]:
len_expts = 180 #200
len_start = 0.005
len_span = 4
sweep_other_param = {
    "length": {
        "label": "probe_pulse",
        "param": "length",
        "param_type": "pulse",
        "expts": len_expts,
        "start": len_start,
        "step": len_span/ len_expts,
        "parent_dict": "probe_pulse_param"
        #"parent_dict": "probe_pulse_param"
        
    },
    }

In [ ]:
qspec =  do_pulse_probe_spec(
    config_thisrun,
    center= 2061, #3063, #config_thisrun.device.qubit.f_ge - 1,
    span=50,
    expts=80, #300,
    reps=500,
    rounds=1,
    prepulse={"pi_qubit_ge":{}, "pi_qubit_ef":{}},  # should be a dict
    
    probe_pulse_param = {
    "gain": 0.5,
    "length": 1.0, #0.03?
    "freq": None,  # doesn't matter
    "chan": config_thisrun.hw.soc.dacs.qubit.ch,
    "sigma": 0,
    "sigma_inc": 0,
    "ramp_sigma": None,
    "ramp_sigma_inc": None,
    "phase":0,
    "type": 'const',},
    analyze_and_display = False,
    sweep_other_param = sweep_other_param
)

In [ ]:
3492.43 + 3369.277 - 4800

In [ ]:
xlist = qspec.data['xpts_freq']
ylist = qspec.data['xpts_gain']
# ylist = qspec.data['xpts_length']
zlists = [qspec.data['avgi'], qspec.data['avgq']]
# print('zlist shape: ', np.array(zlists).shape)
# print(zlists)
zlists = [z.T for z in zlists]
# print('zlist shape after transpose: ', np.array(zlists).shape)
# print(zlists)
zlists_adjusted = [z[:-1, :-1] for z in zlists]  # Adjust dimensions for shading='flat'
freq_guess = config_thisrun.device.qubit.f_ge
print(zlists_adjusted)
plot = meas.ColorPlot2D(xlist = xlist, ylist = ylist,
                        zlists = zlists_adjusted,
                        xlabel = 'freq', ylabel = 'gain ')
              #    xlabel = 'freq', ylabel = 'len') #, title = 'T2 Ramsey Scan')
plot.display() #vlines=[], hlines = [0.5, 1.5, 2, 2.5])

#### Finding coupler

In [ ]:
qspec =  do_pulse_probe_spec(
    config_thisrun,
    center= 4550,
    span=2000,
    expts=500000,
    reps=300,
    rounds=1,
    prepulse= {}, 
    
    probe_pulse_param = {
    "gain": 0.7, #config_thisrun.device.qubit.pulses.pi_ge.gain, #0.123,
    "length": 0.8, #config_thisrun.device.qubit.pulses.pi_ge.sigma *4, #0.2
    "freq": None,  # doesn't matter
    "chan": config_thisrun.hw.soc.dacs.qubit.ch, #send thru cavity pin
    "sigma": 0,
    "sigma_inc": 0,
    "ramp_sigma": None,
    "ramp_sigma_inc": None,
    "phase":0,
    "type": 'const',},
    analyze_and_display = False
)

spec_analysis = meas.SpectroscopyFitting(station = station, data=qspec.data)
spec_analysis.analyze() #fitparams = [-0.7, -0.5, 3064.7, 10]) #[y0, yscale, x0, xscale]
spec_analysis.display()

In [ ]:
qspec =  do_pulse_probe_spec(
    config_thisrun,
    center= 4800,
    span=400,
    expts=20000,
    reps=400,
    rounds=1,
    prepulse= {}, 
    
    probe_pulse_param = {
    "gain": 0.7, #config_thisrun.device.qubit.pulses.pi_ge.gain, #0.123,
    "length": 0.8, #config_thisrun.device.qubit.pulses.pi_ge.sigma *4, #0.2
    "freq": None,  # doesn't matter
    "chan": config_thisrun.hw.soc.dacs.qubit.ch, #send thru cavity pin
    "sigma": 0,
    "sigma_inc": 0,
    "ramp_sigma": None,
    "ramp_sigma_inc": None,
    "phase":0,
    "type": 'const',},
    analyze_and_display = False
)

spec_analysis = meas.SpectroscopyFitting(station = station, data=qspec.data)
spec_analysis.analyze() #fitparams = [-0.7, -0.5, 3064.7, 10]) #[y0, yscale, x0, xscale]
spec_analysis.display()

### T2 Ramsey

In [ ]:
def do_t2_ramsey_ge(
    cfg_dict = cfg_dict,
    config_thisrun = config_thisrun,
    prepulse = {}, # should be a dict, see resonator spec for example of format
    start=0.01,
    step_size=0.02,
    expts=201,
    ramsey_freq=3,  # [MHz]
    reps=200,
    rounds=1,
    relax_delay=200,
        sigma=0.05,
        sigma_inc=0,
        freq=None,
        gain=100,
        pulse_type="gauss",
        analyze_and_display=True,
):
    """Run the T2 Ramsey experiment with all config params as function arguments."""
    t2ramsey = meas.RamseyExperiment(
        cfg_dict=cfg_dict, prefix='RamseyExperiment', go = False,
    )

    t2ramsey.cfg = AttrDict(deepcopy(config_thisrun))
    checkEF = False
    # qubit_ge_init = False
    # qubit_ge_after = False
    # if if_ef:
    #     checkEF = True
    #     qubit_ge_init = True if ef_init else False
    #     qubit_ge_after = True if ef_init else False

    t2ramsey.cfg.expt = {
        'start': start,
        'step': step_size,
        'expts': expts,
        'ramsey_freq': ramsey_freq,
        'reps': reps,
        'rounds': rounds,
        'checkEF': checkEF,
        'prepulse': prepulse,
        
        
        # actual pulse parameters
        "sigma": sigma,  
        "sigma_inc": sigma_inc,
        "freq": freq,
        "gain": gain ,
        "phase": 0,  # First pulse has zero phase
        "type": pulse_type,
        "wait_time": 0.0,
        "num_pi":1,
        "sweep_other_param":{},
    }
    t2ramsey.go(analyze=False, display=False, progress=True, save=True)
    if not analyze_and_display:
        return t2ramsey
    t2ramsey_analysis = meas.RamseyFitting(
        t2ramsey.data, config=t2ramsey.cfg,
    )
    t2ramsey_analysis.display()
    return t2ramsey_analysis



def update_t2_ramsey_ge(t2ramsey, config_thisrun):
    """Update the configuration based on T2 Ramsey experiment results."""
    config_thisrun.device.qubit.f_ge = [config_thisrun.device.qubit.f_ge[0] + min(t2ramsey.data['f_adjust_ramsey_avgi'])]
    
    print('Updated qubit ge frequency to:', config_thisrun.device.qubit.f_ge[0])



In [ ]:
config_thisrun.device.qubit.pulses

In [ ]:
t2ramsey_ge = None
if expts_to_run['t2_ge']:
    # pre_sweep_pulse = [
    #     ['qubit', 'ge', 'pi', 0],
    #     ['qubit', 'ef', 'pi', 0],
    #     ['man', 'M1', 'pi', 0],
    # ]
    pre_sweep_pulse = None
    t2ramsey_ge = do_t2_ramsey_ge(config_thisrun = config_thisrun,
                                  cfg_dict=cfg_dict,
                                  ramsey_freq=0.6,
                                  freq = 3704.9,
                                  gain = config_thisrun.device.qubit.pulses.hpi_ge.gain,
                                  sigma = config_thisrun.device.qubit.pulses.hpi_ge.sigma,
                                  start = 0.01,
                                  step_size=0.02,
                                  sigma_inc = 4,
                                  expts=500,
                                  reps=500)
    # t2ramsey_ge.analyze()
    # t2ramsey_ge.display(title_str='T2_ge')
    print('T2 Ramsey done!')

In [ ]:
t2ramsey_ge.analyze(fitparams=[None, 0.5, None, None, None, None]) #fitparams = yscale, freq, phase_deg, decay, y0, x0 
t2ramsey_ge.display(title_str='T2_ge')#, ylim = [-2.5, -0.5])

In [ ]:
if expts_to_run['t2_ge']:
    update_t2_ramsey_ge(t2ramsey_ge, config_thisrun)
    print('T2 Ramsey updated!')

In [ ]:
config_thisrun.device.qubit.f_ge = 3704.9


In [ ]:
station.handle_config_update(True)

#### T2 ramsey custom

In [ ]:
# pre_sweep_pulse = None
t2ramsey_ge = do_t2_ramsey_ge(config_thisrun = config_thisrun,
                                cfg_dict=cfg_dict,
                                ramsey_freq=0.8,
                                freq = 3492.4,
                                gain = config_thisrun.device.qubit.pulses.hpi_ge.gain,
                                sigma = config_thisrun.device.qubit.pulses.hpi_ge.sigma,
                                start = 0.0,
                                step_size=0.05,
                                sigma_inc = 4,
                                expts=200,
                                reps=500)

#### T2 Ramsey frequency sweep

In [ ]:
ramseys = []
freqs = np.arange(0.0, 1.0, 0.1)
for ramsey_freq in freqs:
    print(f'Running T2 Ramsey with ramsey freq = {ramsey_freq} MHz')
    t2ramsey_ge = do_t2_ramsey_ge(config_thisrun = config_thisrun,
                                  cfg_dict=cfg_dict,
                                  ramsey_freq=ramsey_freq,
                                  freq = config_thisrun.device.qubit.f_ge,#+ramsey_freq,
                                  gain = config_thisrun.device.qubit.pulses.hpi_ge.gain,
                                  sigma = config_thisrun.device.qubit.pulses.hpi_ge.sigma,
                                  start = 0.0,
                                  step_size=0.02,
                                  sigma_inc = 4,
                                  expts=100,
                                  reps=300)
    ramseys.append(t2ramsey_ge)


In [ ]:
def do_single_shot(
    config_thisrun,
    cfg_dict,
    expt_path,
    config_path,
    qubits=[0],
    reps=5000,
    check_f=False,
    active_reset=True,
    man_reset=False,
    storage_reset=False,
    qubit=0,
    pulse_manipulate=False,
    cavity_freq=4984.373226159381,
    cavity_gain=400,
    cavity_length=2,
    prepulse=False,
    pre_sweep_pulse=None,
    gate_based=True,
    relax_delay=500
):
    """Run the single shot experiment with configurable parameters."""

    hstgrm = meas.single_qubit.single_shot.HistogramExperiment(
        soccfg=soc, path=expt_path, prefix='HistogramExperiment', config_file=config_path
    )

    hstgrm.cfg = AttrDict(deepcopy(config_thisrun))

    hstgrm.cfg.expt = {
        'qubits': qubits,
        'reps': reps,
        'check_f': check_f,
        'active_reset': active_reset,
        'man_reset': man_reset,
        'storage_reset': storage_reset,
        'qubit': qubit,
        'pulse_manipulate': pulse_manipulate,
        'cavity_freq': cavity_freq,
        'cavity_gain': cavity_gain,
        'cavity_length': cavity_length,
        'prepulse': prepulse,
        'pre_sweep_pulse': pre_sweep_pulse,
        'gate_based': gate_based,
    }

    hstgrm.cfg.device.readout.relax_delay = [relax_delay]  # Wait time between experiments [us]
    hstgrm.go(analyze=False, display=False, progress=True, save=True)

    from multimode_expts.fit_display_classes import Histogram

    hist_analysis = Histogram(
        hstgrm.data, verbose=True,
        span=None, threshold=None, config=hstgrm.cfg,
    )
    return hstgrm, hist_analysis

In [ ]:
print("xlist shape:", len(xlist))
print("ylist shape:", len(ylist))
print("zlists shapes:", [z.shape for z in zlists_adjusted])

In [ ]:
print("xlist shape:", len(ramseys[0].data['xpts']))
print("ylist shape:", len(freqs))
print("zlists shapes:", [np.array(r.data['avgi']).shape for r in ramseys])

In [ ]:
[r.data['avgi'] for r in ramseys]

In [ ]:
len(ramseys[0].data['avgi'])

In [ ]:
def do_t2_ramsey_ge_sweep(
    cfg_dict = cfg_dict,
    config_thisrun = config_thisrun,
    prepulse = {}, # should be a dict, see resonator spec for example of format
    start=0.01,
    step_size=0.02,
    expts=201,
    ramsey_freq=3,  # [MHz]
    reps=200,
    rounds=1,
    relax_delay=200,
        sigma=0.05,
        sigma_inc=0,
        freq=None,
        gain=100,
        pulse_type="gauss",
    sweep_other_param = {}
):
    """Run the T2 Ramsey experiment with all config params as function arguments."""
    t2ramsey = meas.RamseyExperiment(
        cfg_dict=cfg_dict, prefix='RamseyExperiment', go = False,
    )

    t2ramsey.cfg = AttrDict(deepcopy(config_thisrun))
    checkEF = False
    # qubit_ge_init = False
    # qubit_ge_after = False
    # if if_ef:
    #     checkEF = True
    #     qubit_ge_init = True if ef_init else False
    #     qubit_ge_after = True if ef_init else False

    t2ramsey.cfg.expt = {
        'start': start,
        'step': step_size,
        'expts': expts,
        'ramsey_freq': ramsey_freq,
        'reps': reps,
        'rounds': rounds,
        'checkEF': checkEF,
        'prepulse': prepulse,
        
        
        # actual pulse parameters
        "sigma": sigma,  
        "sigma_inc": sigma_inc,
        "freq": freq,
        "gain": gain ,
        "phase": 0,  # First pulse has zero phase
        "type": pulse_type,
        "wait_time": 0.0,
        "num_pi":1,
        "sweep_other_param": sweep_other_param,
    }
    t2ramsey.go(analyze=False, display=False, progress=True, save=True)
    # if not analyze_and_display:
    #     return t2ramsey
    # t2ramsey_analysis = meas.RamseyFitting(
    #     t2ramsey.data, config=t2ramsey.cfg,
    # )
    # t2ramsey_analysis.display()
    return t2ramsey

In [ ]:
freq = config_thisrun.device.qubit.f_ge
span = 2.0
expts = 80
sweep_other_param={"freq": {"label": "pi2_prep", "param": "freq", "param_type":"pulse", "expts": expts, "start": freq - span/2, "step":span/(expts)}}
t2ramsey_ge_freq_zero = do_t2_ramsey_ge_sweep(config_thisrun = config_thisrun,
                                  cfg_dict=cfg_dict,
                                  ramsey_freq=0.0,
                                  freq = config_thisrun.device.qubit.f_ge,
                                  gain = config_thisrun.device.qubit.pulses.hpi_ge.gain,
                                  sigma = config_thisrun.device.qubit.pulses.hpi_ge.sigma,
                                  start = 0.6,
                                  step_size=0.1,
                                  sigma_inc = 4,
                                  expts=100,
                                  reps=1000, 
                                  sweep_other_param=sweep_other_param)

t2ramsey_ge_freq_two = do_t2_ramsey_ge_sweep(config_thisrun = config_thisrun,
                                  cfg_dict=cfg_dict,
                                  ramsey_freq=0.2,
                                  freq = config_thisrun.device.qubit.f_ge,
                                  gain = config_thisrun.device.qubit.pulses.hpi_ge.gain,
                                  sigma = config_thisrun.device.qubit.pulses.hpi_ge.sigma,
                                  start = 0.6,
                                  step_size=0.1,
                                  sigma_inc = 4,
                                  expts=100,
                                  reps=1000, 
                                  sweep_other_param=sweep_other_param)                    

In [ ]:
ylist = t2ramsey_ge_freq_two.data['xpts_wait_time']
xlist = t2ramsey_ge_freq_two.data['xpts_freq']
zlists = [t2ramsey_ge_freq_two.data['avgi'], t2ramsey_ge_freq_two.data['avgq']]
zlists_adjusted = [z[:-1, :-1] for z in zlists]  # Adjust dimensions for shading='flat'
freq_guess = config_thisrun.device.qubit.f_ge

# print('zlist shape: ', zlist.shape)
plot = meas.ColorPlot2D(xlist = xlist, ylist = ylist,
                        zlists = zlists_adjusted,
                 ylabel = 'Delay (us)', xlabel = 'Ramsey freq (MHz)')#, title = 'T2 Ramsey Scan')
plot.display(vlines = [freq_guess])

In [ ]:
t2ramsey_ge.data.keys()

In [ ]:
xlist

In [ ]:
ylist = t2ramsey_ge.data['xpts_wait_time']
xlist = t2ramsey_ge.data['xpts_freq']
zlists = [t2ramsey_ge.data['avgi'], t2ramsey_ge.data['avgq']]
zlists_adjusted = [z[:-1, :-1] for z in zlists]  # Adjust dimensions for shading='flat'

# print('zlist shape: ', zlist.shape)
plot = meas.ColorPlot2D(xlist = xlist, ylist = ylist,
                        zlists = zlists_adjusted,
                 ylabel = 'Delay (us)', xlabel = 'Ramsey freq (MHz)')#, title = 'T2 Ramsey Scan')
plot.display()

In [ ]:
# Assuming `z` is the 2x2 matrix
# Calculate (max - min) / 2 and add it to each element of `z`
zs = zlists
zadjustsed = []
for z in zs:
    
    z_max_min_half = (np.max(z) - np.min(z)) / 2
    z_adjusted = z + z_max_min_half
    zadjustsed.append(z_adjusted)
    

# Print the adjusted matrix
print("Original Matrix:")
print(z)
print("\nAdjusted Matrix:")
print(z_adjusted.shape)

In [ ]:
# Assuming `z` is the 2D array representing the color plot data
# Collapse the y-axis by multiplying all rows in each column
collapsed_data = np.prod(zs[1][:,:20], axis=0)

# Plot the collapsed data
plt.figure(figsize=(10, 6))
plt.plot(collapsed_data, label='Collapsed Data')
plt.xlabel('X-axis Label')
plt.ylabel('Collapsed Y-axis Product')
plt.title('Collapsed Y-axis of Color Plot')
plt.legend()
plt.grid()
plt.show()

We should probably use a cosine fit with fixed phase=0 instead of decaying sine with varying phase?# Subtract 1.5 from each element of zlist
zlist = zlist - 1.5

### Amplitiude Rabi

In [ ]:
def do_rabi(
    config_thisrun,
    cfg_dict = None,
    start=0.01,
    expts=151,
    reps=200,
    rounds=1,
    sigma_test=None,
    
    pulse_type='gauss',
    sweep='amp',  # 'amp' or 'length'
    chan = 0, 
    freq=3500,  # [MHz]
    gain=0.1,
    sigma=0.1,  # [us] for gauss and flat_top
    sigma_inc=4,  
    length=1.0,  # [us] for const and flat_top
    ramp_sigma=0.02,  # [us] for flat_top
    ramp_sigma_inc=0.0,  # [us] for flat_top
    n_pulses = 1,
    prepulse = {}, # should be a dict, see resonator spec for example of format
    postpulse = {},
    
    max_gain=1.0,  # Max gain for amplitude sweep
    max_length=10.0,  # Max length for length sweep
    if_ef=False,  # If true, will check ef frequency and update its
    sweep_other_param = {}
):
    """
    Run the Rabi experiment.
    All experiment parameters are function arguments.
    """
    
    rabi = meas.RabiExperiment(
        cfg_dict=cfg_dict, prefix='RabiExperiment', go = False,
    )
    rabi.cfg = AttrDict(deepcopy(config_thisrun))

    # pulse_ge = config_thisrun.device.qubit.pulses.pi_ge
    # if sigma_test is None:
    #     sigma_test = pulse_ge.sigma[0]
    # if start is None:
    #     start = 0
    # if step is None:
    #     step = int(pulse_ge.gain[0] / (expts - 1))
    
    # checkEF = False
    # pulse_ge_init = False
    # pulse_ge_after = False
    # if if_ef:
    #     checkEF = True
    #     pulse_ge_init = True
    #     pulse_ge_after = True

    rabi.cfg.expt = dict(
        start=start,
        max_gain = max_gain,
        max_length = max_length,
        expts=expts,
        reps=reps,
        rounds=rounds,
        sigma_test=sigma_test,
        
        # pulse parameters
        sweep = sweep, # amp or length
        chan  = chan,
        freq = freq,
        type = pulse_type, 
        gain = gain,
        sigma = sigma,
        step = (max_gain - start)/expts,
        sigma_inc = sigma_inc, 
        length = length,
        ramp_sigma = ramp_sigma,
        ramp_sigma_inc = ramp_sigma_inc,
        prepulse = prepulse,
        postpulse = postpulse,
        n_pulses = n_pulses,
        final_delay = config_thisrun.device.readout.final_delay,
        sweep_other_param = sweep_other_param
    )

    rabi.go(analyze=False, display=False, progress=True, save=True)

    rabi_analysis = meas.AmplitudeRabiFitting(
        rabi.data, 
        readout_per_round=4, config=rabi.cfg, station = station
    )
    return rabi_analysis

def update_amplitude_rabi(amprabi, config_thisrun):
    """Update the configuration based on amplitude Rabi experiment results."""
    
    config_thisrun.device.qubit.pulses.pi_ge.gain = float(np.round(amprabi.data['pi_gain_avgi'],3))
    config_thisrun.device.qubit.pulses.hpi_ge.gain = float(np.round(amprabi.data['hpi_gain_avgi'],3))
    # also the sigmas
    config_thisrun.device.qubit.pulses.pi_ge.sigma = amprabi.cfg.expt.sigma
    config_thisrun.device.qubit.pulses.hpi_ge.sigma = amprabi.cfg.expt.sigma
    station.handle_config_update(True)
    print('Updated qubit ge pi and hpi gaussian gain!')
    

def update_amplitude_rabi_ef(amprabi, config_thisrun):
    """Update the configuration based on amplitude Rabi experiment results."""
    
    config_thisrun.device.qubit.pulses.pi_ef.gain = float(np.round(amprabi.data['pi_gain_avgi'],3))
    config_thisrun.device.qubit.pulses.hpi_ef.gain = float(np.round(amprabi.data['hpi_gain_avgi'],3))
    # also the sigmas
    config_thisrun.device.qubit.pulses.pi_ef.sigma = amprabi.cfg.expt.sigma
    config_thisrun.device.qubit.pulses.hpi_ef.sigma = amprabi.cfg.expt.sigma
    station.handle_config_update(True)
    print('Updated qubit ef pi and hpi gaussian gain!')

In [ ]:
# config_thisrun.device.readout.frequency = 7604.36
# config_thisrun.device.readout.gain = 0.08
# config_thisrun.device.readout.length = 3
# config_thisrun.device.readout.final_delay = 250
# station.handle_config_update(True)



In [ ]:
amprabi = do_rabi(cfg_dict = cfg_dict,
                  config_thisrun=config_thisrun,
                  sweep = 'amp',
                  start = 0.001,
                            expts=300, reps=500, max_gain = 0.8, 
                            gain = 0.0, # ignore
                            chan = config_thisrun.hw.soc.dacs.qubit.ch, 
                            freq= config_thisrun.device.qubit.f_ge, 
                            sigma = 0.03,
                            length = 0.0,
                            sigma_inc = 4,
                            prepulse = {},#{"pi_qubit_ge":{}}, #,"pi_qubit_ef":{}}, #{'hpi_qubit_ge': {}},
                            pulse_type='gauss')

In [ ]:
amprabi.analyze() #fitparams = [2.5, 2, -90, None, -2.5, None]) # yscale, freq, phase_deg, decay, y0, x0
amprabi.display()#hlines=[-0.8, -1.2])

In [ ]:
update_amplitude_rabi(amprabi, config_thisrun)

In [ ]:
t2_ramsey_ge_after_amp = None
if expts_to_run['amplitude_ge']:
    # After this do another round of T2 to fine tune the qubit frequency
    t2_ramsey_ge_after_amp  = do_t2_ramsey_ge(config_thisrun, expt_path, config_path,
                                              ramsey_freq=2.0,
                                              step_size=soc.cycles2us(8),
                                              expts=100,
                                              reps=100, active_reset=False, relax_delay=2500)
    t2_ramsey_ge_after_amp.analyze()
    t2_ramsey_ge_after_amp.display(title_str='T2_ge_after_amp')
    update_t2_ramsey_ge(t2_ramsey_ge_after_amp , config_thisrun)
    print('T2 Ramsey done!')

In [ ]:
# amprabi.analyze(fitparams= [np.max(amprabi.data['amps']), 0.00001, 90, None, None, None])
# amprabi.display()
# update_amplitude_rabi(amprabi, config_thisrun)
# print('Amplitude Rabi done!')

### Slow Pi pulse

In [ ]:
amprabi = do_rabi(cfg_dict = cfg_dict,
                  config_thisrun=config_thisrun,
                  sweep = 'amp',
                  start = 0.001,
                            expts=151, reps=500, max_gain = 0.5, 
                            gain = 0.0, # ignore
                            chan = config_thisrun.hw.soc.dacs.qubit.ch, 
                            freq=config_thisrun.device.qubit.f_ge, 
                            sigma = 0.05,
                            length = 3.0,
                            sigma_inc = 4,
                            prepulse = {},
                            pulse_type='const')

In [ ]:
amprabi.analyze(fitparams=[np.ptp(amprabi.data['avgi'])/2, 0.0001, np.pi, 1000000,None, None])
amprabi.display(title_str='Amplitude Rabi_ge', save_fig=True, 
                # ylim = [-2.5, -0.5])
)

In [ ]:
# amprabi.analyze(fitparams= [np.max(amprabi.data['amps']), 0.0001, 90, None, None, None])
# amprabi.display()
# update_amplitude_rabi(amprabi, config_thisrun)


# Magic Params: [np.max(amprabi.data['amps']), 0.00001, 90, None, None, None]

#### Sweep Frequency to find the largest contrast

In [ ]:
freq_expts = 30
freq_span = 3
freq_start = config_thisrun.device.qubit.f_ge - freq_span/2



sweep_other_param = {"freq": {
        "label": "rabi_pulse",
        "param": "freq",
        "param_type": "pulse",
        "expts": freq_expts,
        "start": freq_start,
        "step": freq_span/ freq_expts
    },}

amprabi = do_rabi(cfg_dict = cfg_dict,
                  config_thisrun=config_thisrun,
                  sweep = 'amp',
                  start = 0.001,
                            expts=151, reps=100, max_gain = 1.0, 
                            gain = 0.0, # ignore
                            chan = config_thisrun.hw.soc.dacs.qubit.ch, 
                            freq=config_thisrun.device.qubit.f_ge, 
                            sigma = 0.05,
                            length = 3.0,
                            sigma_inc = 4,
                            prepulse = {},
                            pulse_type='const', 
                            sweep_other_param = sweep_other_param)

In [ ]:
amprabi.data.keys()

In [ ]:
amprabi.data['avgi'].shape

In [ ]:
# Define keys for amplitude Rabi experiment
keys = ['xpts_gain', 'xpts_freq']

# Extract data for plotting
ylist = amprabi.data[keys[0]]
xlist = amprabi.data[keys[1]]
zlists = [amprabi.data['avgi'], amprabi.data['avgq']]

# Adjust dimensions for shading='flat'
zlists_adjusted = [z[:-1, :-1] for z in zlists]

# Create and display the 2D color plot
plot = meas.ColorPlot2D(
    xlist=xlist,
    ylist=ylist,
    zlists=zlists_adjusted,
    ylabel=keys[0],
    xlabel=keys[1]
)

plot.display()

### T1

In [ ]:
def do_t1_ge(
    cfg_dict=cfg_dict,
    config_thisrun=config_thisrun,
    prepulse={},       # should be a dict, see resonator spec for example of format
    start=0.01,
    step_size=0.5,
    expts=201,
    reps=200,
    rounds=1,
    relax_delay=200,
    sigma=None,
    sigma_inc=None,
    freq=None,
    gain=None,
    pulse_type=None,
    analyze_and_display=True,
):
    """Run the T1 experiment with all config params as function arguments.
    
    Pulse sequence: π pulse → wait_time → measure
    Fits exponential decay to extract T1.

    Args:
        cfg_dict:             QICK config dictionary
        config_thisrun:       Full config for this run (deepcopied internally)
        prepulse:             Optional prepulse dict (same format as resonator spec)
        start:                Wait time start [us]
        step_size:            Wait time step [us] — set so span ~3xT1
        expts:                Number of wait time points
        reps:                 Number of averages per experiment
        rounds:               Number of software average rounds
        relax_delay:          Relax delay between shots [us]
        sigma:                Pulse sigma [us]
        sigma_inc:            Pulse sigma increment
        freq:                 Qubit drive frequency [MHz] (None = use config default)
        gain:                 Pulse gain
        pulse_type:           Pulse shape ('gauss', 'flat_top', etc.)
        analyze_and_display:  If True, fits and displays results. If False, returns raw experiment.

    Returns:
        T1Fitting object (if analyze_and_display=True) or raw T1Experiment object
    """
    t1 = meas.T1Experiment(
        cfg_dict=cfg_dict, prefix='T1Experiment', go=False,
    )
    t1.cfg = AttrDict(deepcopy(config_thisrun))
    if freq is None: 
        freq = config_thisrun.device.qubit.f_ge
    if gain is None:
        gain = config_thisrun.device.qubit.pulses.pi_ge.gain
    if sigma is None:
        sigma = config_thisrun.device.qubit.pulses.pi_ge.sigma
    if pulse_type is None:
        pulse_type = 'gauss'
    if sigma_inc is None:
        sigma_inc = 4
        

    t1.cfg.expt = {
        'start': start,
        'step': step_size,
        'expts': expts,
        'reps': reps,
        'rounds': rounds,
        'prepulse': prepulse,

        # pulse parameters
        "sigma": sigma,
        "sigma_inc": sigma_inc,
        "freq": freq,
        "gain": gain,
        "phase": 0,
        "type": pulse_type,
        "wait_time": 0.0,
        "sweep_other_param": {},
    }
    t1.go(analyze=False, display=False, progress=True, save=True)

    if not analyze_and_display:
        return t1

    t1_analysis = meas.T1Fitting(
        t1.data, config=t1.cfg,
    )
    t1_analysis.display()
    return t1_analysis

In [ ]:
t1 = None
if expts_to_run['t1_ge']:
    t1 = do_t1_ge()
    t1.analyze()
    t1.display()
    # update_t1_ge(t1, config_thisrun)
    print('T1 done!')

## Qubit ef

### Pulse-probe

In [ ]:
def do_pulse_probe_ef(config_thisrun): 

    qspec = meas.single_qubit.pulse_probe_ef_spectroscopy.PulseProbeEFSpectroscopyExperiment(
        soccfg=soc, path=expt_path, prefix='PulseProbeEFSpectroscopyExperiment', config_file=config_path
    )

    qspec.cfg = AttrDict(deepcopy(config_thisrun))

    qspec.cfg.expt = {'start': 3415,
        'step': 0.05,
        'expts': 500,
        'reps': 200,
        'rounds': 1,
        'length': 1,
        'gain': 100,
        # 'pulse_type': 'gaussian',
        'qubit_f': False,
        'qubit': 0,
        'cavity_drive': False,
        'wait_qubit': False,}



    qspec.cfg.device.readout.relax_delay = [500] # Wait time between experiments [us]
    qspec.go(analyze=True, display=True, progress=True, save=True)
    return qspec

def update_pulse_probe_ef(qspec, config_thisrun):
    config_thisrun.device.qubit.f_ef = [qspec.data['fit_avgi'][2]]
    print('Updated qubit frequency!')

In [ ]:
# expts_to_run['pulse_probe_ef'] = True

In [ ]:
qspec_ef = None
if expts_to_run['pulse_probe_ef']: 
    qspec_ef = do_pulse_probe_ef(config_thisrun)


In [ ]:
if expts_to_run['pulse_probe_ef']:
    update_pulse_probe_ef(qspec_ef, config_thisrun)
    print('Pulse probe spectroscopy done!')

### T2 Ramsey

In [ ]:
def do_t2_ramsey_ef(config_thisrun, expt_path, config_path, ef_init = True, pre_sweep_pulse = None, post_sweep_pulse = None, ramsey_freq = 3, step_size = soc.cycles2us(8), 
                    active_reset = False, relax_delay = 2500, reps = 100):
    """Run the T2 Ramsey experiment."""
    return do_t2_ramsey_ge(config_thisrun, expt_path, config_path, pre_sweep_pulse=pre_sweep_pulse, 
                            post_sweep_pulse=post_sweep_pulse, ramsey_freq=ramsey_freq, step_size=step_size, if_ef=True, ef_init=ef_init,
                            active_reset=active_reset, relax_delay=relax_delay, reps = reps)
    


def update_t2_ramsey_ef(t2ramsey, config_thisrun):
    """Update the configuration based on T2 Ramsey experiment results."""
    config_thisrun.device.qubit.f_ef = [config_thisrun.device.qubit.f_ef[0] + min(t2ramsey.data['f_adjust_ramsey_avgi'])]
    print('Updated qubit ef frequency to:', config_thisrun.device.qubit.f_ef[0])



In [ ]:
t2ramsey_ef = None
if expts_to_run['t2_ef']:
    t2ramsey_ef = do_t2_ramsey_ef(config_thisrun, expt_path, config_path)
    t2ramsey_ef.analyze()
    t2ramsey_ef.display(title_str='T2_ef')
    # update_t2_ramsey_ef(t2ramsey_ef, config_thisrun)
    print('T2 Ramsey done!')

In [ ]:
# t2ramsey_ef.analyze(fitparams=[100, 10, None, 20, None, None])
# t2ramsey_ef.display(title_str='T2_ef')
update_t2_ramsey_ef(t2ramsey_ef, config_thisrun)

### Amplitude Rabi

We should probably use a cosine fit with fixed phase=0 instead of decaying sine with varying phase?

In [ ]:
def update_amplitude_rabi_ef(amprabi_ef, config_thisrun):
    """Update the configuration based on amplitude Rabi experiment results for the ef transition."""
    config_thisrun.device.qubit.pulses.pi_ef.gain = float(np.round(amprabi_ef.data['pi_gain_avgi'],3))
    config_thisrun.device.qubit.pulses.hpi_ef.gain = float(np.round(amprabi_ef.data['hpi_gain_avgi'],3))
    # also the sigmas
    config_thisrun.device.qubit.pulses.pi_ef.sigma = amprabi_ef.cfg.expt.sigma
    config_thisrun.device.qubit.pulses.hpi_ef.sigma = amprabi_ef.cfg.expt.sigma
    station.handle_config_update(True)
    print('Updated qubit ef pi and hpi gaussian gain!')


In [ ]:
amprabi = do_rabi(cfg_dict = cfg_dict,
                  config_thisrun=config_thisrun,
                  sweep = 'amp',
                  start = 0.001,
                            expts=151, reps=500, max_gain = 1.0, 
                            gain = 0.0, # ignore
                            chan = config_thisrun.hw.soc.dacs.qubit.ch, 
                            freq= config_thisrun.device.qubit.f_ef, 
                            sigma = 0.03,
                            length = 0.0,
                            sigma_inc = 4,
                            prepulse = {"pi_qubit_ge":{}}, #{'hpi_qubit_ge': {}},
                            postpulse = {"pi_qubit_ge":{}},
                            pulse_type='gauss')

In [ ]:
amprabi.analyze()#fitparams = [2.5, 2, -90, None, -2.5, None]) # yscale, freq, phase_deg, decay, y0, x0
amprabi.display(hlines=[])

In [ ]:
update_amplitude_rabi_ef(amprabi, config_thisrun)

In [ ]:
amprabi_ef = None
t2_ramsey_ef_after_amp = None
if expts_to_run['amplitude_ef']:
    amprabi_ef = do_amplitude_rabi_ef(config_thisrun, expt_path, config_path)
    amprabi_ef.analyze(fitparams=[np.max(amprabi_ef.data['amps']), 0.0001, np.pi, 10000,None, None])
    amprabi_ef.display(title_str = 'Amplitude Rabi_ef', save_fig=True)
    update_amplitude_rabi_ef(amprabi_ef, config_thisrun)
    print('Amplitude Rabi done!')

    # After this do another round of T2 to fine tune the qubit frequency
    t2_ramsey_ef_after_amp  = do_t2_ramsey_ef(config_thisrun, expt_path, config_path)
    t2_ramsey_ef_after_amp.analyze()
    t2_ramsey_ef_after_amp.display(title_str='T2_ef_after_amp')
    update_t2_ramsey_ef(t2_ramsey_ef_after_amp , config_thisrun)
    print('T2 Ramsey done!')

In [ ]:
# amprabi_ef.analyze(title_str = 'Amplitude Rabi_ef', save_fig=False, fitparams=[np.max(amprabi_ef.data['amps']), 0.0001, 90, None, None, None])
# amprabi_ef.display(title_str = 'Amplitude Rabi_ef', save_fig=False)
# Magic Params: [np.max(amprabi_ef.data['amps']), 0.00001, 90, None, None, None]

In [ ]:
# update_amplitude_rabi_ef(amprabi_ef, config_thisrun)
# print('Amplitude Rabi done!')

# # After this do another round of T2 to fine tune the qubit frequency
# t2_ramsey_ef_after_amp  = do_t2_ramsey_ef(config_thisrun, expt_path, config_path)
# t2_ramsey_ef_after_amp.analyze()
# t2_ramsey_ef_after_amp.display(title_str='T2_ef_after_amp')
# update_t2_ramsey_ef(t2_ramsey_ef_after_amp , config_thisrun)
# print('T2 Ramsey done!')

In [ ]:
# t2_ramsey_ef_after_amp.analyze(fitparams=[300, None, None, None, None, None])
# t2_ramsey_ef_after_amp.display(title_str='T2_ef_after_amp')
# update_t2_ramsey_ef(t2_ramsey_ef_after_amp , config_thisrun)

### T1

In [ ]:
def do_t1_ef(config_thisrun, expt_path, config_path):
    """Run the T1 experiment."""
    t1 = meas.single_qubit.t1.T1Experiment(
        soccfg=soc, path=expt_path, prefix='T1Experiment', config_file=config_path
    )

    t1.cfg = AttrDict(deepcopy(config_thisrun))

    t1.cfg.expt = {
        'start': 0,
        'step': 5,
        'expts': 100,
        'reps': 50,
        'rounds': 1,
        'qubit': 0,
        'qubit_ef': True,
        'normalize': False
    }

    t1.cfg.device.readout.relax_delay = [2500]  # Wait time between experiments [us]
    t1.go(analyze=True, display=True, progress=True, save=True)
    return t1


def update_t1_ef(t1, config_thisrun):
    """Update the configuration based on T1 experiment results."""
    config_thisrun.device.qubit.T1_ef = [t1.data['fit_avgq'][3]]
    print('Updated qubit T1!')


In [ ]:
t1_ef = None
if expts_to_run['t1_ef']:
    t1 = do_t1_ef(config_thisrun, expt_path, config_path)
    t1.analyze()
    t1.display()
    update_t1_ef(t1, config_thisrun)
    print('T1 done!')

# Cavity Drive

## Manipulate and Cavity Spec

In [ ]:
cav_freqs = [7494.320, 7588.199] #7387.680, 7494.320, 7588.199, 7599.485,    7524.936, 7526, 7911, 7913.200]
 
for freq in cav_freqs:
    rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, 
                    frequency = freq, span = 0.2, reps = 200, expts = 2000, gain=0.7, final_delay=250)
    max_idx = rspec.data['amps'].argmax()
    fwhm_idx = np.where(rspec.data['amps'] >= rspec.data['amps'][max_idx]/2)
    print(f"Frequency at max amplitude: {rspec.data['xpts'][max_idx]} MHz")
    print(f"Max amplitude value: {rspec.data['amps'][max_idx]}")
    print(f"FWHM at amplitude: {rspec.data['amps'][fwhm_idx[0][0]]} and {rspec.data['amps'][fwhm_idx[0][-1]]}")
    print(f"Freq of FWHM at: {rspec.data['xpts'][fwhm_idx[0][0]]} and {rspec.data['xpts'][fwhm_idx[0][-1]]} MHz")
    spec_analysis = meas.SpectroscopyFitting(data=rspec.data, station = station)
    spec_analysis.analyze()
    spec_analysis.display()
    # spec_analysis.display(title = 'Cavity Spectroscopy', hlines=[rspec.data['amps'][fwhm_idx[0][0]]], vlines=[rspec.data['xpts'][fwhm_idx[0][0]], rspec.data['xpts'][fwhm_idx[0][-1]]])

In [ ]:
man_freqs = [3009.4, 3065, 3216.800, 3319.4, 3437.8, 3625.2, 3856.576, 4088, 4780.4, 4808.2693] 
cav_freqs = [4976.206800000, 5110, 5138.249900000, 5300.958000000, 5578.8, 5622.688300000, 5786.404600000, 5969.312200000, 6026.113, 6141, 6130.556458100, 6322.852481800, 6391.3, 6512.205951000, 6708.674069300, 6849.800, 7201.06, 7316.76, 7517.5, 7671.240]

# for freq in man_freqs:
#     rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, 
#                     frequency = freq, span = 10, reps = 400, expts = 4000, gain=0.7, length = 30, final_delay=250)
#     max_idx = rspec.data['amps'].argmax()
#     fwhm_idx = np.where(rspec.data['amps'] >= rspec.data['amps'][max_idx]/2)
#     print(f"Frequency at max amplitude: {rspec.data['xpts'][max_idx]} MHz")
#     print(f"Max amplitude value: {rspec.data['amps'][max_idx]}")
#     print(f"FWHM at amplitude: {rspec.data['amps'][fwhm_idx[0][0]]} and {rspec.data['amps'][fwhm_idx[0][-1]]}")
#     print(f"Freq of FWHM at: {rspec.data['xpts'][fwhm_idx[0][0]]} and {rspec.data['xpts'][fwhm_idx[0][-1]]} MHz")
#     spec_analysis = meas.SpectroscopyFitting(data=rspec.data, station = station)
#     spec_analysis.analyze()
#     spec_analysis.display(title = 'Manipulate Spectroscopy', hlines=[rspec.data['amps'][fwhm_idx[0][0]]], vlines=[rspec.data['xpts'][fwhm_idx[0][0]], rspec.data['xpts'][fwhm_idx[0][-1]]])
 
for freq in cav_freqs:
    rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, 
                    frequency = freq, span = 10, reps = 400, expts = 4000, gain=0.7, length = 30, final_delay=250)
    max_idx = rspec.data['amps'].argmax()
    fwhm_idx = np.where(rspec.data['amps'] >= rspec.data['amps'][max_idx]/2)
    print(f"Frequency at max amplitude: {rspec.data['xpts'][max_idx]} MHz")
    print(f"Max amplitude value: {rspec.data['amps'][max_idx]}")
    print(f"FWHM at amplitude: {rspec.data['amps'][fwhm_idx[0][0]]} and {rspec.data['amps'][fwhm_idx[0][-1]]}")
    print(f"Freq of FWHM at: {rspec.data['xpts'][fwhm_idx[0][0]]} and {rspec.data['xpts'][fwhm_idx[0][-1]]} MHz")
    spec_analysis = meas.SpectroscopyFitting(data=rspec.data, station = station)
    spec_analysis.analyze()
    spec_analysis.display(title = 'Cavity Spectroscopy', hlines=[rspec.data['amps'][fwhm_idx[0][0]]], vlines=[rspec.data['xpts'][fwhm_idx[0][0]], rspec.data['xpts'][fwhm_idx[0][-1]]])

In [ ]:
cav_freqs = [6027, 6323, 7387.7, 7493.3, 7524.9, 7588.2, 7599.5, 7914.2]

for freq in cav_freqs:
    rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, 
                    frequency = freq, span = 5, reps = 400, expts = 1000, gain=0.7, length = 30, final_delay=250)
    # max_idx = rspec.data['amps'].argmax()
    # fwhm_idx = np.where(rspec.data['amps'] >= rspec.data['amps'][max_idx]/2)
    # print(f"Frequency at max amplitude: {rspec.data['xpts'][max_idx]} MHz")
    # print(f"Max amplitude value: {rspec.data['amps'][max_idx]}")
    # print(f"FWHM at amplitude: {rspec.data['amps'][fwhm_idx[0][0]]} and {rspec.data['amps'][fwhm_idx[0][-1]]}")
    # print(f"Freq of FWHM at: {rspec.data['xpts'][fwhm_idx[0][0]]} and {rspec.data['xpts'][fwhm_idx[0][-1]]} MHz")
    spec_analysis = meas.SpectroscopyFitting(data=rspec.data, station = station)
    spec_analysis.analyze()
    spec_analysis.display(title = 'Cavity Spectroscopy')
    # spec_analysis.display(title = 'Cavity Spectroscopy', hlines=[rspec.data['amps'][fwhm_idx[0][0]]], vlines=[rspec.data['xpts'][fwhm_idx[0][0]], rspec.data['xpts'][fwhm_idx[0][-1]]])

## Circle fits


### imports + connect to VNA:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sys 
import os
print(os.getcwd())
project_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
project_path2 = os.path.abspath(os.path.join(os.getcwd(), '..', 'scresonators'))
project_path3 = os.path.abspath(os.path.join(os.getcwd(), '..', 'scresonators', 'measurement'))
print(project_path, project_path2, project_path3) 
sys.path.append(project_path)
sys.path.append(project_path2) #add root path to sys.path to import fit_resonator
sys.path.append(project_path3)
print(sys.path)

from datamanagement import SlabFile
import datetime
from time import *
# from slab import *
from handy import prev_data
from fit_resonator.ana_resonator import ResonatorFitter

# # #new_path = r'/Users/sph/Library/CloudStorage/GoogleDrive-circuitqed@gmail.com/My Drive/Projects/Materials/materials-software/'
# # new_path = r'G:/My Drive/Projects/Materials/materials-software/'
# # sys.path.append(new_path)


%load_ext autoreload
%autoreload 2
import vna_measurement
from ZNB import ZNB20


setup + connect to vna: \
port 1 - C3 (cav readout) \
port 2 - A6 (input) \
port 3 - A4 (cav drive) \
port 4 - C1 (readout) 

In [ ]:
VNA = ZNB20(address='10.108.30.65') # Stanford
#VNA = ZNB20(address='192.168.137.84') # SLAC

# Daily measurement settings
warm_att = 0 #dB
cold_att = 60

# base_path = r'G:/My Drive/Projects/Materials/Data/multimode_cav_8_15_25'
# if not os.path.exists(base_path):
#     os.makedirs(base_path)

# spar = 's11'
base_path = r'C:/Users/Administrator/Documents/multimode_expts_tprocv2/scresonators/measurement'
if not os.path.exists(base_path):
    os.makedirs(base_path)

### low power Qs:

In [ ]:
%matplotlib inline
expt_path = base_path
qplots_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'measurement_notebooks'))

nb_points = 10001
span=6e6
span2=0.1e6
span3 = 0.5e6
bandwidth= 500
avgs= 10
power = -30

# freqs = np.array([3.81958, 4.81469])*1e9 
# freqs = np.array([7494.3])*1e6
freqs = np.array([7494.083, 7525.226])*1e6
freqs2 = np.array([6026.141 , 6322.951, 7387.698, 7599.608])*1e6
freqs3 = np.array([7588.205])*1e6

output_arr_lowp = []
for freq_center in freqs:    
    scan_def = {'freq_center':freq_center, 'span':span, 'bandwidth':bandwidth, 'power':power, 'npoints':int(nb_points), 'averages':avgs, 'spar': 's13'}
    file_name = 'res_' + str(freq_center)[:7] + '_' + str(power)[1:] + 'dbm' + '_' + str(bandwidth) + 'bw_' + datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    data = vna_measurement.do_vna_scan(VNA, file_name, expt_path, scan_def)
    output = ResonatorFitter.fit_resonator(data, str(file_name), qplots_path, plot=True, fix_freq=False, fit_type='DCM REFLECTION'
            )
    output_arr_lowp += [output]
    
    
for freq_center in freqs2:    
    scan_def = {'freq_center':freq_center, 'span':span2, 'bandwidth':bandwidth, 'power':power, 'npoints':int(nb_points), 'averages':avgs, 'spar': 's13'}
    file_name = 'res_' + str(freq_center)[:7] + '_' + str(power)[1:] + 'dbm' + '_' + str(bandwidth) + 'bw_' + datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    data = vna_measurement.do_vna_scan(VNA, file_name, expt_path, scan_def)
    output = ResonatorFitter.fit_resonator(data, str(file_name), qplots_path, plot=True, fix_freq=False, fit_type='DCM REFLECTION'
            )
    output_arr_lowp += [output]

for freq_center in freqs3:    
    scan_def = {'freq_center':freq_center, 'span':span3, 'bandwidth':bandwidth, 'power':power, 'npoints':int(nb_points), 'averages':avgs, 'spar': 's13'}
    file_name = 'res_' + str(freq_center)[:7] + '_' + str(power)[1:] + 'dbm' + '_' + str(bandwidth) + 'bw_' + datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    data = vna_measurement.do_vna_scan(VNA, file_name, expt_path, scan_def)
    output = ResonatorFitter.fit_resonator(data, str(file_name), qplots_path, plot=True, fix_freq=False, fit_type='DCM REFLECTION'
            )
    output_arr_lowp += [output]

In [ ]:
# Calculate arrays based on output_arr
q_arr = [output_arr_lowp[i][0][0] for i in range(len(output_arr_lowp))]
qc_arr = [output_arr_lowp[i][0][1] for i in range(len(output_arr_lowp))]
f_arr = [output_arr_lowp[i][0][2] for i in range(len(output_arr_lowp))]
qi_arr = [(q_arr[i]**-1 - qc_arr[i]**-1)**-1 for i in range(len(q_arr))]

# Overall scatter plot of qc_arr and qi_arr
plt.scatter(f_arr, qc_arr, label='Qc (All)')
plt.scatter(f_arr, qi_arr, label='Qi (All)')
plt.yscale('log')  # Set the y-axis to logarithmic scale
plt.grid(which='both', linestyle='--', linewidth=0.5, color='gray')  # Add grid lines
plt.legend()
plt.title('Qc and Qi (All)')
plt.xlabel('Frequency (f)')
plt.ylabel('Value')
plt.show()

### High power Qs

In [ ]:
%matplotlib inline
expt_path = base_path
qplots_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'measurement_notebooks'))

nb_points = 10001
span=6e6
span2=0.1e6
span3 = 0.5e6
bandwidth=500
avgs= 10
power = 0

# freqs = np.array([3.81958, 4.81469])*1e9 
# freqs = np.array([7494.3])*1e6
freqs = np.array([7493.326, 7524.950])*1e6
freqs2 = np.array([6026.114, 6322.857, 7387.680, 7599.4805])*1e6
freqs3 = np.array([7588.199])*1e6

output_arr_highp = []
for freq_center in freqs:    
    scan_def = {'freq_center':freq_center, 'span':span, 'bandwidth':bandwidth, 'power':power, 'npoints':int(nb_points), 'averages':avgs, 'spar': 's13'}
    file_name = 'res_' + str(freq_center)[:7] + '_' + str(power)[1:] + 'dbm' + '_' + str(bandwidth) + 'bw_' + datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    data = vna_measurement.do_vna_scan(VNA, file_name, expt_path, scan_def)
    output = ResonatorFitter.fit_resonator(data, str(file_name), qplots_path, plot=True, fix_freq=False, fit_type='DCM REFLECTION'
            )
    output_arr_highp += [output]
    
for freq_center in freqs2:    
    scan_def = {'freq_center':freq_center, 'span':span2, 'bandwidth':bandwidth, 'power':power, 'npoints':int(nb_points), 'averages':avgs, 'spar': 's13'}
    file_name = 'res_' + str(freq_center)[:7] + '_' + str(power)[1:] + 'dbm' + '_' + str(bandwidth) + 'bw_' + datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    data = vna_measurement.do_vna_scan(VNA, file_name, expt_path, scan_def)
    output = ResonatorFitter.fit_resonator(data, str(file_name), qplots_path, plot=True, fix_freq=False, fit_type='DCM REFLECTION'
            )
    output_arr_highp += [output]

for freq_center in freqs3:    
    scan_def = {'freq_center':freq_center, 'span':span3, 'bandwidth':bandwidth, 'power':power, 'npoints':int(nb_points), 'averages':avgs, 'spar': 's13'}
    file_name = 'res_' + str(freq_center)[:7] + '_' + str(power)[1:] + 'dbm' + '_' + str(bandwidth) + 'bw_' + datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    data = vna_measurement.do_vna_scan(VNA, file_name, expt_path, scan_def)
    output = ResonatorFitter.fit_resonator(data, str(file_name), qplots_path, plot=True, fix_freq=False, fit_type='DCM REFLECTION'
            )
    output_arr_highp += [output]

In [ ]:
# Calculate arrays based on output_arr
q_arr = [output_arr_highp[i][0][0] for i in range(len(output_arr_highp))]
qc_arr = [output_arr_highp[i][0][1] for i in range(len(output_arr_highp))]
f_arr = [output_arr_highp[i][0][2] for i in range(len(output_arr_highp))]
qi_arr = [(q_arr[i]**-1 - qc_arr[i]**-1)**-1 for i in range(len(q_arr))]

# Overall scatter plot of qc_arr and qi_arr
plt.scatter(f_arr, qc_arr, label='Qc (All)')
plt.scatter(f_arr, qi_arr, label='Qi (All)')
plt.yscale('log')  # Set the y-axis to logarithmic scale
plt.grid(which='both', linestyle='--', linewidth=0.5, color='gray')  # Add grid lines
plt.legend()
plt.title('Qc and Qi (All)')
plt.xlabel('Frequency (f)')
plt.ylabel('Value')
plt.show()

### Punchouts:

In [ ]:
lowp = [6.026141, 6.322951, 7.494083, 7.525226, 7.588205, 7.387698, 7.599608]
highp = [6.026114, 6.322857, 7.493326, 7.524950, 7.588199, 7.387680, 7.5994805]
for i in range(len(lowp)):
    print((lowp[i] - highp[i])*1e3, 'MHz')

In [ ]:
punchouts = [output_arr_lowp[i][0][2] - output_arr_highp[i][0][2] for i in range(len(output_arr_lowp))]
print(f'{np.array(punchouts)*1e-6} MHz frequency punchouts between low and high power for each resonator')
print(f'{[f"{f*1e-6:.2f}" for f in f_arr]} MHz mode freqs')
plt.scatter(f_arr, punchouts)

## Mode Searching with VNA

In [ ]:
sstart = 4.95e9
sstop = 6.80e9
sspan = 0.1e6 

nb_points = 10001
bandwidth= 500
avgs= 10
power = 0

mode_search = np.arange(sstart, sstop, sspan)

# for power in [-30, 0]:
for mode in mode_search:    
    scan_def = {'freq_center':mode, 'span':sspan*1.2, 'bandwidth':bandwidth, 'power':power, 'npoints':int(nb_points), 'averages':avgs, 'spar': 's13'}
    file_name = 'res_' + str(mode)[:7] + '_' + str(power)[1:] + 'dbm' + '_' + str(bandwidth) + 'bw_' + datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    data = vna_measurement.do_vna_scan(VNA, file_name, expt_path, scan_def)

In [ ]:
sstart = 5.0e9
sstop = 6.80e9
sspan = 0.2e6 

nb_points = 10001
bandwidth= 500
avgs= 10
power = 0

mode_search = np.arange(sstart, sstop, sspan)

# for power in [-30, 0]:
for mode in mode_search:    
    scan_def = {'freq_center':mode, 'span':sspan*1.1, 'bandwidth':bandwidth, 'power':power, 'npoints':int(nb_points), 'averages':avgs, 'spar': 's13'}
    file_name = 'res_' + str(mode)[:7] + '_' + str(power)[1:] + 'dbm' + '_' + str(bandwidth) + 'bw_' + datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    data = vna_measurement.do_vna_scan(VNA, file_name, expt_path, scan_def)

## Mode Searching with RFSoC

In [ ]:
sstart = 5.00e9
sstop = 6.80e9
sspan = 0.2e6 


mode_search = np.arange(sstart, sstop, sspan)
print(mode_search)
print(len(mode_search)/60/60)

for freq in mode_search:
    rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, 
                    frequency = freq, span = 0.22, reps = 300, expts = 500, gain=0.7, length = 20, final_delay=250)
    max_idx = rspec.data['amps'].argmax()
    # fwhm_idx = np.where(rspec.data['amps'] >= rspec.data['amps'][max_idx]/2)
    print(f"Frequency at max amplitude: {rspec.data['xpts'][max_idx]} MHz")
    print(f"Max amplitude value: {rspec.data['amps'][max_idx]}")
    # print(f"FWHM at amplitude: {rspec.data['amps'][fwhm_idx[0][0]]} and {rspec.data['amps'][fwhm_idx[0][-1]]}")
    # print(f"Freq of FWHM at: {rspec.data['xpts'][fwhm_idx[0][0]]} and {rspec.data['xpts'][fwhm_idx[0][-1]]} MHz")
    spec_analysis = meas.SpectroscopyFitting(data=rspec.data, station = station)
    spec_analysis.analyze()
    spec_analysis.display(title='Mode Searching')

# Manipulate

## F0g1 spec using qubit spec

In [ ]:
f0g1spec =  do_pulse_probe_spec(
    config_thisrun,
    center= 3064,
    span=10,
    expts=400,
    reps=1000,
    rounds=1,
    prepulse= {"pi_qubit_ge":{}, "pi_qubit_ef":{}},  # should be a dict
    
    
    probe_pulse_param = {
    "gain": 0.5, #0.123,
    "length": 2, #0.2
    "freq": None,  # doesn't matter
    "chan": config_thisrun.hw.soc.dacs.qubit.ch,
    "sigma": 0,
    "sigma_inc": 0,
    "ramp_sigma": None,
    "ramp_sigma_inc": None,
    "phase":0,
    "type": 'const',
    "t":0},
    analyze_and_display = False
)

In [ ]:
spec_analysis = meas.SpectroscopyFitting(station = station, data=f0g1spec.data)
spec_analysis.analyze() #fitparams = [-0.7, -0.5, 3064.7, 10]) #[y0, yscale, x0, xscale]
spec_analysis.display()

## Spectroscopy

In [ ]:
def do_pulse_probe_f0g1(config_thisrun, ds_thisrun, man_mode_no = 1): 

    qspec = meas.single_qubit.pulse_probe_f0g1_spectroscopy.PulseProbeF0g1SpectroscopyExperiment(
        soccfg=soc, path=expt_path, prefix='PulseProbeF0g1SpectroscopyExperiment', config_file=config_path
    )

    qspec.cfg = AttrDict(deepcopy(config_thisrun))

    qspec.cfg.expt = {
        'start': ds_thisrun.get_freq('M' + str(man_mode_no)) - 10,  # resonator frequency to be mixed up [MHz]
        'step': 0.1,  # min step ~1 MHz
        'expts': 200,  # Number of experiments stepping from start
        'reps': 100,  # Number of averages per point
        'rounds': 1,  # Number of start to finish sweeps to average over
        'length': 1,  # ef probe constant pulse length [us]
        'gain': 3000,  # f0g1 pulse gain
        'pulse_type': 'gaussian',
        'qubit_f': True,
        'qubits': [0],
        'prepulse': False,
        # 'pre_sweep_pulse': [[3569.4827896982997], [11161], [0], [0], [2], ['g'], [0.035]]
    }

    qspec.cfg.device.readout.relax_delay = [200] # Wait time between experiments [us]
    qspec.go(analyze=False, display=False, progress=True, save=True)
    return qspec

def analyze_and_display_pulse_probe_f0g1(qspec):
    from multimode_expts.fit_display_classes import Spectroscopy
    spec_analysis = Spectroscopy(
        qspec.data)
    spec_analysis.analyze(fit=True)
    spec_analysis.display()

def update_pulse_probe_f0g1(qspec, config_thisrun, man_mode_no = 1):
    ''' 
    Update the configuration based on f0g1 spectroscopy experiment results.
    man_mode_no: 1 for man1, 2 for man2
    '''
    ds_thisrun.update_freq('M' + str(man_mode_no),  qspec.data['fit_avgi'][2])
    config_thisrun.device.multiphoton['pi']['fn-gn+1']['frequency'][0] = qspec.data['fit_avgi'][2]


    print('Updated man f0g1 freq to :', ds_thisrun.get_freq('M1'))


In [ ]:
# expts_to_run['pulse_probe_f0g1'] = True

man_specs =[None]* len(expts_to_run['man_modes'])

for i in range(len(expts_to_run['man_modes'])):

    if expts_to_run['pulse_probe_f0g1']:

        print('Running pulse probe f0g1 for mode', i+1)
        man_specs[i] = do_pulse_probe_f0g1(config_thisrun, ds_thisrun, man_mode_no = i+1)
        analyze_and_display_pulse_probe_f0g1(man_specs[i])
        update_pulse_probe_f0g1(man_specs[i], config_thisrun, man_mode_no=i+1)
        
        

## Find Frequency (Chevron)


In [ ]:
def do_length_rabi_f0g1_sweep(config_thisrun, expt_path, config_path, freq_start, freq_stop, freq_step):
    """Run the Length Rabi General F0g1 Experiment Sweep."""
    # length_rabi = meas.single_qubit.length_rabi.LengthRabiGeneralF0g1ExperimentSweep(
    #     soccfg=soc, path=expt_path, prefix='LengthRabiGeneralF0g1ExperimentSweep', config_file=config_path
    # )
    from multimode_expts.sequential_experiment_classes import man_f0g1_class
    experiment_class = man_f0g1_class
    sweep_experiment_name = 'length_rabi_f0g1_sweep'
    class_for_exp = experiment_class(soccfg=soc, path=expt_path, prefix=sweep_experiment_name, config_file=config_path,
                                      exp_param_file=exp_param_file, config_thisrun=config_thisrun)

    class_for_exp.yaml_cfg = AttrDict(deepcopy(config_thisrun))
    
    class_for_exp.loaded[sweep_experiment_name] = {
        'freq_start': freq_start,
        'freq_stop':  freq_stop,
        'freq_step': freq_step,
        'start': 2,
        'step': 0.1,
        'qubits': [0],
        'expts': 25,
        'reps': 100,
        'rounds': 1,
        'gain': 8000,
        'ramp_sigma': 0.005,
        'use_arb_waveform': False,
        'pi_ge_before': True,
        'pi_ef_before': True,
        'pi_ge_after': False,
        'normalize': False,
        'active_reset': False,
        'check_man_reset': [False, 0],
        'check_man_reset_pi': [],
        'prepulse': False,
        'pre_sweep_pulse': [],
        'err_amp_reps' : 0, # Number of error amp rounds
    } # actually this doesn't do anything, edit experiment.yml
    
 
    return eval('class_for_exp.run_sweep')( sweep_experiment_name = sweep_experiment_name)

def update_length_rabi_f0g1_sweep(expt_path, prefix, config_thisrun, man_mode_no = 1):
    """Update sweep data and analyze results."""
    temp_data, attrs, filename = prev_data(expt_path, prefix=prefix)
    print('File saved at:', filename)
    from multimode_expts.fit_display_classes import ChevronFitting
    from datetime import datetime
    
    chevron_analysis = ChevronFitting(
        frequencies=temp_data['freq_sweep'],
        time=temp_data['xpts'][0],
        response_matrix=temp_data['avgi'],
        config = config_thisrun
    )
    chevron_analysis.analyze()
    current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    chevron_analysis.display_results(save_fig=True,  title=f'M{man_mode_no}_{current_time}')

    #config_thisrun.device.manipulate.f0g1_freq[0] = chevron_analysis.results['frequency']
    ds_thisrun.update_freq('M' + str(man_mode_no), chevron_analysis.results['best_frequency_contrast'])
    print('Updated the frequency to:', chevron_analysis.results['best_frequency_contrast'])
    return chevron_analysis


In [ ]:
ds_thisrun.get_freq('M' + str(0 + 1))

In [ ]:
man_chevrons_coarse = [None] * len(expts_to_run['stor_modes'])
man_chevrons_fine = [None] * len(expts_to_run['stor_modes'])
# man_sweeps = [None] * len(expts_to_run['stor_modes'])

for i in range(len(expts_to_run['stor_modes'])):
    if expts_to_run['length_rabi_sweep']:
        # print(f'Running coarse length rabi sweep for mode {i + 1}')
        # freq_start = ds_thisrun.get_freq('M' + str(i + 1)) - 3
        # freq_stop = ds_thisrun.get_freq('M' + str(i + 1)) + 3
        # freq_step = 0.3
        # man_chevrons_coarse[i] = do_length_rabi_f0g1_sweep(config_thisrun, expt_path, config_path, freq_start, freq_stop, freq_step)
        # update_length_rabi_f0g1_sweep(expt_path, 'length_rabi_f0g1_sweep', config_thisrun, man_mode_no=i + 1)

        print(f'Running fine length rabi sweep for mode {i + 1}')
        freq_start = ds_thisrun.get_freq('M' + str(i + 1)) - 0.3
        freq_stop = ds_thisrun.get_freq('M' + str(i + 1)) + 0.3
        # freq_start = 2017
        # freq_stop = 2021
        freq_step = 0.025
        # freq_step = 1.
        man_chevrons_fine[i] = do_length_rabi_f0g1_sweep(config_thisrun, ds_thisrun, expt_path, config_path, freq_start, freq_stop, freq_step)
        update_length_rabi_f0g1_sweep(expt_path, 'length_rabi_f0g1_sweep', config_thisrun, man_mode_no=i + 1)



## Error amplification

In [ ]:
from MM_dual_rail_base import MM_dual_rail_base
mm_base_calib = MM_dual_rail_base(config_thisrun, ds_thisrun)

sideband = 'f0-g1'
_sideband = sideband[0] + 'n' + '-' + sideband[3] + 'n+1'
i = int(sideband[1])
pre_sweep_pulse = mm_base_calib.prep_man_photon(man_no=1, photon_no=i)
print(f"pre_sweep_pulse: {pre_sweep_pulse}")
pre_sweep_pulse.append(['multiphoton', 'g'+str(i)+'-e'+str(i), 'pi', 0])
pre_sweep_pulse.append(['multiphoton', 'e'+str(i)+'-f'+str(i), 'pi', 0])
# pre_sweep_pulse = mm_base_calib.get_prepulse_creator(pre_sweep_pulse, multiphoton_cfg_thisrun).pulse.tolist()
pre_sweep_pulse = mm_base_calib.get_prepulse_creator(pre_sweep_pulse, config_thisrun).pulse.tolist()
# print(f"pre_sweep_pulse: {pre_sweep_pulse}")

In [ ]:
def do_error_amplification(
    config_thisrun,
    expt_path,
    config_path,
    reps=100,
    rounds=1,
    qubit=0,
    n_pulses=10,
    active_reset=False,
    man_reset=True,
    storage_reset=True,
    relax_delay=2500, 
    start = 0,
    expts = 10,
    step = 100,
    parameter_to_test = 'gain',
    pulse_type=['qubit', 'ge', 'pi', 0],
):
    """
    Run the Histogram Prepulse Experiment with configurable parameters.
    """
        
    expt_cfg = {
        'reps': reps,
        'qubit': qubit,
        'qubits': [qubit],
        'active_reset': active_reset,
        'man_reset': man_reset,
        'storage_reset': storage_reset,
        'start': start,
        'expts': expts,
        'step': step,
        'n_pulses': n_pulses,
        'pulse_type': pulse_type,
        'parameter_to_test': parameter_to_test,
        'rounds': rounds,

    }
    error_amp_exp = meas.single_qubit.error_amplification.ErrorAmplificationExperiment(
    soccfg=soc, path=expt_path,
      prefix='ErrorAmplificationExperiment', config_file=config_path)
    error_amp_exp.cfg = AttrDict(deepcopy(config_thisrun))
    error_amp_exp.cfg.expt = expt_cfg
    error_amp_exp.go(analyze=False, display=False, progress=True, save=True)
    return error_amp_exp

In [ ]:
config_thisrun.device.multiphoton['pi'][_sideband]['frequency'][i]

In [ ]:
expts = 20
# band = 0.75
band = 0.5
# config_thisrun.device.multiphoton['pi'][_sideband]['length'][0] = 0.5491206701698654
freq_start = config_thisrun.device.multiphoton['pi'][_sideband]['frequency'][i] - band
# freq_start = 2011 - band
step = 2 * band / expts
print(f'Frequency start: {freq_start}, Step: {step}')

err_amp_man_freq = do_error_amplification(
    config_thisrun=config_thisrun,
    expt_path=expt_path,
    config_path=config_file,
    reps=100,
    rounds=1,
    n_pulses = 7,
    expts=expts,
    start=freq_start,
    step=step,
    pulse_type=['multiphoton', sideband, 'pi', 0],
    parameter_to_test='frequency',
)

In [ ]:
err_amp_man_freq.analyze(state_fin='e')
err_amp_man_freq.display()

In [ ]:
print(f"Man {sideband} pi frequency before update:", config_thisrun.device.multiphoton['pi'][_sideband]['frequency'][i])
config_thisrun.device.multiphoton['pi'][_sideband]['frequency'][i] = err_amp_man_freq.data['fit_avgi'][2]
print(f"Man {sideband} pi frequency after update:", config_thisrun.device.multiphoton['pi'][_sideband]['frequency'][i])

In [ ]:
if i > 0:
    print("WARNING! No update will occur! The update in this cell was meant for the csv which does not have multiphoton params. To update the multiphoton params, please run the multiphoton calibration notebook instead.")
else:
    ds_thisrun.update_freq('M1', err_amp_man_freq.data['fit_avgi'][2])
    print("Updated the ds_thisrun frequency to:", ds_thisrun.get_freq('M1'))

## Length Rabi f0g1 (Update time)

In [ ]:
def do_length_rabi_f0g1_general(config_thisrun, ds_thisrun, expt_path, config_path, man_mode_no = 1):
    """Run the Length Rabi General F0g1 Experiment."""
    length_rabi = meas.single_qubit.length_rabi_f0g1_general.LengthRabiGeneralF0g1Experiment(
        soccfg=soc, path=expt_path, prefix='LengthRabiGeneralF0g1Experiment', config_file=config_path
    )

    length_rabi.cfg = AttrDict(deepcopy(config_thisrun))

    length_rabi.cfg.expt = dict(
        start=soc.cycles2us(3),  # Pulse start length [us]
        step=0.01,  # Pulse step length [us]
        qubits=[0],
        expts=150,
        reps=100,
        rounds=1,
        # rep_start=0, # 0 means just 1 pi pulse, no error amp
        # rep_end=19, # will do 1+2*rep_end rounds of pi pulses max
        gain=8000,  # Qubit gain [DAC units]
        freq= ds_thisrun.get_freq('M' + str(man_mode_no)),  # Frequency [MHz]
        use_arb_waveform=False,
        pi_ge_before=True,
        pi_ef_before=True,
        pi_ge_after=True,
        # pi_ef_after=True,
        normalize=False,
        active_reset=False,
        man_reset=True,
        stor_reset=True,
        check_man_reset=[False, 0],
        swap_lossy=False,
        check_man_reset_pi=[],
        prepulse=False,
        pre_sweep_pulse=[],
        err_amp_reps = 0, # Number of error amp rounds
    )

    length_rabi.cfg.device.readout.relax_delay = [5000]  # Wait time between experiments [us]
    length_rabi.go(analyze=False, display=False, progress=True, save=True)

    from multimode_expts.fit_display_classes import LengthRabiFitting
    # Analyze the data
    length_rabi_analysis = LengthRabiFitting(length_rabi.data, config = length_rabi.cfg)
    length_rabi_analysis.analyze()
    length_rabi_analysis.display(title_str='Length Rabi General F0g1')


    return length_rabi_analysis


def update_length_rabi_f0g1_combined(length_rabi, config_thisrun, ds, man_mode_no = 1):
    """Update the configuration and dataset based on Length Rabi General F0g1 experiment results."""

    # Update dataset
    pi_length = length_rabi.results['pi_length']
    pi2_length = length_rabi.results['pi2_length']
    gain = length_rabi.cfg.expt['gain']
    freq = length_rabi.cfg.expt['freq']
    ds.update_all('M' + str(man_mode_no), freq, np.nan, pi_length, pi2_length, gain)
    print(f'Updated dataset: pi_length={pi_length}, pi2_length={pi2_length}, gain={gain}')



In [ ]:
len_rabis_mans = [None]* len(expts_to_run['man_modes'])
for i in range(len(expts_to_run['man_modes'])):
    if expts_to_run['length_rabi'] or expts_to_run['length_rabi_sweep']:
        print('Running length rabi for mode', i+1)
        len_rabis_mans[i] = do_length_rabi_f0g1_general(config_thisrun, 
                                                        ds_thisrun, 
                                                        expt_path, 
                                                        config_path, 
                                                        man_mode_no= i+1)
        update_length_rabi_f0g1_combined(len_rabis_mans[i], config_thisrun, ds_thisrun, man_mode_no= i+1)

In [ ]:
# len_rabis_mans[0].active_reset = True
# len_rabis_mans[0].analyze()
# len_rabis_mans[0].display(title_str='Length Rabi General F0g1')

### Quick and dirty error amplification

Very slightly modified length rabi f0g1 general (see git)

In [ ]:
def do_error_amplification(
    config_thisrun,
    ds_thisrun,
    expt_path,
    config_path,
    reps=100,
    rounds=1,
    qubit=0,
    n_pulses=10,
    active_reset=False,
    man_reset=True,
    storage_reset=True,
    relax_delay=2500, 
    start = 0,
    expts = 10,
    step = 100,
    parameter_to_test = 'gain',
    pulse_type=['qubit', 'ge', 'pi', 0],
):
    """
    Run the Histogram Prepulse Experiment with configurable parameters.
    """
        
    expt_cfg = {
        'reps': reps,
        'qubit': qubit,
        'qubits': [qubit],
        'active_reset': active_reset,
        'man_reset': man_reset,
        'storage_reset': storage_reset,
        'start': start,
        'expts': expts,
        'step': step,
        'n_pulses': n_pulses,
        'pulse_type': pulse_type,
        'parameter_to_test': parameter_to_test,
        'rounds': rounds,

    }

    error_amp_exp = meas.single_qubit.error_amplification.ErrorAmplificationExperiment(
    soccfg=soc, path=expt_path,
      prefix='ErrorAmplificationExperiment', config_file=config_path)
    error_amp_exp.cfg = AttrDict(deepcopy(config_thisrun))
    error_amp_exp.cfg.expt = expt_cfg
    error_amp_exp.go(analyze=False, display=False, progress=True, save=True)
    return error_amp_exp

In [ ]:
expts_to_run['stor_modes'] = [1, 2, 3, 4, 5, 6, 7]
# expts_to_run['stor_modes'] = [5]

sideband_chevrons_coarse = [None] * len(expts_to_run['stor_modes'])
sideband_chevrons_fine = [None] * len(expts_to_run['stor_modes'])
error_amp_gain1 = [None] * len(expts_to_run['stor_modes'])
error_amp_freq1 = [None] * len(expts_to_run['stor_modes'])
error_amp_gain2 = [None] * len(expts_to_run['stor_modes'])
error_amp_freq2 = [None] * len(expts_to_run['stor_modes'])

In [ ]:
for i, stor_i in enumerate(expts_to_run['stor_modes']):
# for stor_i in [5]:

    stor_name = 'M1-S' + str(stor_i)
    if expts_to_run['sideband_freq_sweep']:

        # ds_thisrun.update_gain(stor_name, 9000)
        # ds_thisrun.update_freq(stor_name, 873.6)
        # ds_thisrun.update_freq(stor_name, 348.66845603890465)
        # ds_thisrun.update_pi(stor_name, abs(ds_thisrun.get_pi(stor_name)))
        # ds_thisrun.update_h_pi(stor_name, abs(ds_thisrun.get_h_pi(stor_name)))

        print(f'Running coarse sideband sweep for storage mode {stor_i}')
        freq_start = ds_thisrun.get_freq(stor_name) - 1.5
        freq_stop = ds_thisrun.get_freq(stor_name) + 1.5
        freq_step = 0.2
        # freq_start = ds_thisrun.get_freq(stor_name) - 0.2
        # freq_stop = ds_thisrun.get_freq(stor_name) + 0.2
        # freq_step = 0.1
        sideband_chevrons_coarse[i] = do_sideband_general_sweep(config_thisrun, ds_thisrun, expt_path, config_path, freq_start, freq_stop, freq_step, reps=50, man_mode_no=1, stor_mode_no=stor_i, liveplotting=False)
        update_sideband_general_sweep(expt_path, config_thisrun, ds_thisrun, man_mode_no=1, stor_mode_no=stor_i)


        print(f'Running error amplification (gain) for storage mode {stor_i}')
        error_amp_exp_gain = do_error_amp_storage(
            config_thisrun, ds_thisrun, expt_path, config_path, reps=50, rounds=1, qubit=0, span=1000, expts=50, n_start=0, n_step=3, n_pulses=15, man_mode_no=1, stor_mode_no=stor_i, stor_is_dump=False, parameter_to_test='gain')
        error_amp_gain1[i] = error_amp_exp_gain
        error_amp_exp_gain.analyze(state_fin='e')
        error_amp_exp_gain.display()
        gain_opt = int(error_amp_exp_gain.data['fit_avgi'][2])
        # Update dataset
        ds_thisrun.update_gain(stor_name, gain_opt)
        print('Updated the gain to:', gain_opt)


        print(f'Running error amplification (frequency) for storage mode {stor_i}')
        error_amp_exp_freq = do_error_amp_storage(
            config_thisrun, ds_thisrun, expt_path, config_path, reps=50, rounds=1, qubit=0, span=0.3, expts=50, n_pulses=10, man_mode_no=1, stor_mode_no=stor_i, stor_is_dump=False, parameter_to_test='frequency')
        error_amp_freq1[i] = error_amp_exp_freq
        error_amp_exp_freq.analyze(state_fin='e')
        error_amp_exp_freq.display()
        freq_opt = error_amp_exp_freq.data['fit_avgi'][2]
        # Update dataset
        ds_thisrun.update_freq(stor_name, freq_opt)
        print('Updated the frequency to:', freq_opt)


        print(f'Running error amplification (gain) ROUND 2 for storage mode {stor_i}')
        error_amp_exp_gain2 = do_error_amp_storage(
            config_thisrun, ds_thisrun, expt_path, config_path, reps=50, rounds=1, qubit=0, span=600, expts=50, n_start=1, n_step=1, n_pulses=10, man_mode_no=1, stor_mode_no=stor_i, stor_is_dump=False, parameter_to_test='gain')
        error_amp_gain2[i] = error_amp_exp_gain2
        error_amp_exp_gain2.analyze(state_fin='e')
        error_amp_exp_gain2.display()
        gain_opt = int(error_amp_exp_gain2.data['fit_avgi'][2])
        # Update dataset
        ds_thisrun.update_gain(stor_name, gain_opt)
        print('Updated the gain to:', gain_opt)



In [ ]:
# for i, stor_i in enumerate(expts_to_run['stor_modes']):
#     if stor_i != 4: continue
#     # if stor_i == 5: continue

#     stor_name = 'M1-S' + str(stor_i)
#     print(stor_name)
#     ds.update_gain(stor_name, ds_thisrun.get_gain(stor_name))
#     ds.update_freq(stor_name, ds_thisrun.get_freq(stor_name))
#     ds.update_pi(stor_name, ds_thisrun.get_pi(stor_name))
#     ds.update_h_pi(stor_name, ds_thisrun.get_h_pi(stor_name))

# Storage

## Stor Spectroscopy

In [ ]:
def get_storage_mode_parameters(ds_thisrun, config_thisrun, man_mode_no, stor_mode_no):
    """
    Get pulse parameters for a given storage mode. 
    Also returns prepulse and postpulse (single photon prep and meas for ge meas)

    Args:
        ds_thisrun: Dataset object for managing frequency data.
        config_thisrun: Configuration dictionary for the current run.
        man_mode_no: Manipulation mode number.
        stor_mode_no: Storage mode number.

    Returns:
        A tuple containing freq, gain, ch, prepulse, and postpulse.
    """
    stor_name = 'M' + str(man_mode_no) + '-S' + str(stor_mode_no)
    freq = ds_thisrun.get_freq(stor_name)
    gain = ds_thisrun.get_gain(stor_name)
    pi_len = ds_thisrun.get_pi(stor_name)
    h_pi_len = ds_thisrun.get_h_pi(stor_name)
    flux_low_ch = config_thisrun.hw.soc.dacs.flux_low.ch
    flux_high_ch = config_thisrun.hw.soc.dacs.flux_high.ch
    ch = flux_low_ch if freq<1000 else flux_high_ch

    from MM_dual_rail_base import MM_dual_rail_base
    mm_base_dummy = MM_dual_rail_base(config_thisrun, soccfg=soc)
    prep_man_pi = mm_base_dummy.prep_man_photon(man_mode_no)
    prepulse = mm_base_dummy.get_prepulse_creator(prep_man_pi).pulse.tolist()
    print("post pulse", prep_man_pi[-1:-3:-1])
    postpulse = mm_base_dummy.get_prepulse_creator(prep_man_pi[-1:-3:-1]).pulse.tolist() # for ge meas, only do f0g1 and ef pi

    return freq, gain, pi_len, h_pi_len, ch, prepulse, postpulse


def do_stor_spectroscopy(config_thisrun, ds_thisrun,  expt_path, config_path, man_mode_no = 1, stor_no = 1):
    """
    Run the Flux Spectroscopy F0g1 Experiment.

    This function performs a flux spectroscopy experiment to measure the transition frequency
    between the f0 and g1 states of a qubit. It configures the experiment parameters, executes
    the experiment, and saves the results.

    Args:
        config_thisrun (AttrDict): Configuration dictionary for the current run.
        ds_thisrun (dataset.storage_man_swap_dataset): Dataset object for managing frequency data.
        expt_path (str): Path to save the experiment results.
        config_path (str): Path to the configuration file.
        man_mode_no (int, optional): Manipulation mode number (default is 1).
        stor_no (int, optional): Storage mode number (default is 1).

    Returns:
        FluxSpectroscopyF0g1Experiment: The experiment object containing the results.
    """
    flux_spec = meas.single_qubit.rf_flux_spectroscopy_f0g1.FluxSpectroscopyF0g1Experiment(
        soccfg=soc, path=expt_path, prefix='FluxSpectroscopyF0g1Experiment', config_file=config_path
    )

    flux_spec.cfg = AttrDict(deepcopy(config_thisrun))

    freq, gain, pi_len, h_pi_len, ch, prepulse, postpulse = get_storage_mode_parameters(ds_thisrun, config_thisrun,
                                                                      man_mode_no, stor_no)
    flux_spec.cfg.expt = dict(
        start=freq - 30,  # Start RF frequency [MHz]
        step=0.3,  # Step size [MHz]
        expts=200,  # Number of experiments
        reps=100,  # Number of averages per point
        qubit=[0],
        flux_drive=[ch, 1, 1000, 5],  # RF flux modulation parameters [low/high (ch), freq (will be overwritten), gain, length(us)]
        prepulse=True,
        postpulse=True,
        active_reset=True,
        pre_sweep_pulse= prepulse,
        post_sweep_pulse= postpulse,
    )

    flux_spec.cfg.device.readout.relax_delay = [500]  # Wait time between experiments [us]
    flux_spec.go(analyze=False, display=False, progress=True, save=True)
    return flux_spec



def analyze_and_display_stor_spectroscopy(flux_spec):
    """Analyze and display results of Flux Spectroscopy F0g1 Experiment."""
    from multimode_expts.fit_display_classes import Spectroscopy
    spec_analysis = Spectroscopy(flux_spec.data)
    spec_analysis.analyze(fit=True)
    spec_analysis.display()


def update_stor_spectroscopy(flux_spec, ds_thisrun, man_mode_no = 1, stor_no = 1):
    """Update the configuration based on Flux Spectroscopy F0g1 experiment results."""
    # Update the dataset with the new frequency
    ds_thisrun.update_freq('M' + str(man_mode_no) + '-S' + str(stor_no), flux_spec.data['fit_avgi'][2])
    print(f"Updated frequency for M{man_mode_no}-S{stor_no}: {flux_spec.data['fit_avgi'][2]}")


In [ ]:
stor_specs = [None]* len(expts_to_run['stor_modes'])
for i in range(len(expts_to_run['stor_modes'])):
    if expts_to_run['stor_spectroscopy']:
        print('Running flux spectroscopy f0g1 for mode', i+1)
        flux_spec = do_stor_spectroscopy(config_thisrun, ds_thisrun, expt_path, config_path, man_mode_no=1, stor_no=i+1)
        stor_specs[i] = flux_spec
        analyze_and_display_stor_spectroscopy(flux_spec)
        update_stor_spectroscopy(flux_spec, ds_thisrun, man_mode_no=1, stor_no=i+1)

In [ ]:
i = 6
flux_spec = do_stor_spectroscopy(config_thisrun, ds_thisrun, expt_path, config_path, man_mode_no=1, stor_no=i+1)
analyze_and_display_stor_spectroscopy(flux_spec)
# update_stor_spectroscopy(flux_spec, ds_thisrun, man_mode_no=1, stor_no=i+1)

In [ ]:
# flux low
i = 5
spec = do_stor_spectroscopy(config_thisrun, ds_thisrun, expt_path, config_path, man_mode_no=1, stor_no=i+1, use_flux_low = True)
analyze_and_display_stor_spectroscopy(spec)

In [ ]:
for i in range(len(expts_to_run['stor_modes'])):
    print('Frequency for M1-S' + str(i+1) + ':', ds_thisrun.get_freq('M1-S' + str(i+1)))


### Man-dump

In [ ]:
from MM_dual_rail_base import MM_dual_rail_base
mm_base_dummy = MM_dual_rail_base(config_thisrun, soccfg=soc)
prep_man_pi = mm_base_dummy.prep_man_photon(man_mode_no)
prepulse = mm_base_dummy.get_prepulse_creator(prep_man_pi).pulse.tolist()
postpulse = mm_base_dummy.get_prepulse_creator(prep_man_pi[-1:-3:-1]).pulse.tolist() # for ge meas, only do f0g1 and ef pi

In [ ]:
def get_dump_mode_parameters(ds_thisrun, config_thisrun, man_mode_no, dump_mode_no):
    """
    Get pulse parameters for a given storage mode. 
    Also returns prepulse and postpulse (single photon prep and meas for ge meas)

    Args:
        ds_thisrun: Dataset object for managing frequency data.
        config_thisrun: Configuration dictionary for the current run.
        man_mode_no: Manipulation mode number.
        dump_mode_no: Dump mode number.

    Returns:
        A tuple containing freq, gain, ch, prepulse, and postpulse.
    """
    stor_name = 'M' + str(man_mode_no) + '-D' + str(dump_mode_no)
    freq = ds_thisrun.get_freq(stor_name)
    gain = ds_thisrun.get_gain(stor_name)
    pi_len = ds_thisrun.get_pi(stor_name)
    h_pi_len = ds_thisrun.get_h_pi(stor_name)
    ch = 'low' if freq < 1000 else 'high'

    from MM_dual_rail_base import MM_dual_rail_base
    mm_base_dummy = MM_dual_rail_base(config_thisrun, soccfg=soc)
    prep_man_pi = mm_base_dummy.prep_man_photon(man_mode_no)
    prepulse = mm_base_dummy.get_prepulse_creator(prep_man_pi).pulse.tolist()
    postpulse = mm_base_dummy.get_prepulse_creator(prep_man_pi[-1:-3:-1]).pulse.tolist() # for ge meas, only do f0g1 and ef pi

    prepulse_overwrite = [['multiphoton', 'g0-e0', 'pi', 0],
                            ['multiphoton', 'e0-f0', 'pi', 0],
                            ['multiphoton', 'f0-g1', 'pi', 0]
                        ]
    postpulse_overwrite = [ ['multiphoton', 'f0-g1', 'pi', 0],
                            ['multiphoton', 'e0-f0', 'pi', 0]]
    prepulse = mm_base_dummy.get_prepulse_creator(prepulse_overwrite).pulse.tolist()
    postpulse = mm_base_dummy.get_prepulse_creator(postpulse_overwrite).pulse.tolist()

    return freq, gain, pi_len, h_pi_len, ch, prepulse, postpulse


def do_dump_spectroscopy(config_thisrun, 
                         ds_thisrun, 
                         expt_path, 
                         config_path, 
                         man_mode_no = 1, 
                         dump_no = 1,
                         flux_gain = 5000,
                         flux_length = 1):
    """
    Run the Flux Spectroscopy F0g1 Experiment.

    This function performs a flux spectroscopy experiment to measure the transition frequency
    between the f0 and g1 states of a qubit. It configures the experiment parameters, executes
    the experiment, and saves the results.

    Args:
        config_thisrun (AttrDict): Configuration dictionary for the current run.
        ds_thisrun (dataset.storage_man_swap_dataset): Dataset object for managing frequency data.
        expt_path (str): Path to save the experiment results.
        config_path (str): Path to the configuration file.
        man_mode_no (int, optional): Manipulation mode number (default is 1).
        dump_no (int, optional): Storage mode number (default is 1).

    Returns:
        FluxSpectroscopyF0g1Experiment: The experiment object containing the results.
    """
    flux_spec = meas.single_qubit.rf_flux_spectroscopy_f0g1.FluxSpectroscopyF0g1Experiment(
        soccfg=soc, path=expt_path, prefix='FluxSpectroscopyF0g1Experiment', config_file=config_path
    )

    flux_spec.cfg = AttrDict(deepcopy(config_thisrun))

    freq, gain, pi_len, h_pi_len, ch, prepulse, postpulse = get_dump_mode_parameters(ds_thisrun, config_thisrun,
                                                                      man_mode_no, dump_no)

    # freq = 420

    flux_spec.cfg.expt = dict(
        start=freq - 100,  # Start RF frequency [MHz]
        step=0.1,  # Step size [MHz]
        expts=50,  # Number of experiments
        reps=200,  # Number of averages per point
        qubit=[0],
        flux_drive=[ch, 1, flux_gain, flux_length],  # RF flux modulation parameters [low/high (ch), freq (will be overwritten), gain, length(us)]
        prepulse=True,
        postpulse=True,
        active_reset=True,
        man_reset=False,
        storage_reset=False,
        pre_sweep_pulse= prepulse,
        post_sweep_pulse= postpulse,
    )

    flux_spec.cfg.device.readout.relax_delay = [200]  # Wait time between experiments [us]
    flux_spec.go(analyze=False, display=False, progress=True, save=True)
    return flux_spec


def update_dump_spectroscopy(flux_spec, ds_thisrun, man_mode_no = 1, dump_no = 1):
    """Update the configuration based on Flux Spectroscopy F0g1 experiment results."""
    # Update the dataset with the new frequency
    ds_thisrun.update_freq('M' + str(man_mode_no) + '-D' + str(dump_no), flux_spec.data['fit'][2])
    print(f"Updated frequency for M{man_mode_no}-D{dump_no}: {flux_spec.data['fit'][2]}")


In [ ]:
spec = do_dump_spectroscopy(config_thisrun, ds_thisrun, expt_path, config_path, 
                            man_mode_no=1, dump_no=1,
                            flux_gain=500, flux_length=5)
# analyze_and_display_stor_spectroscopy(spec)
# update_dump_spectroscopy(spec, ds_thisrun, 1, 1)
spec.analyze(fit=True)
spec.display()

In [ ]:
idata = spec.data['idata']
idata = idata.reshape((len(idata.flatten())//4,4))

qdata = spec.data['qdata']
qdata = qdata.reshape((len(qdata.flatten())//4,4))

fig, axs = plt.subplots(nrows=4,ncols=2, figsize=(8,8))
for kk in range(4):
    axs[kk,0].hist(idata[:,kk], bins=100)
    axs[kk,1].hist(qdata[:,kk], bins=100)
None

In [ ]:
len(idata[:,3]), np.mean(idata[:,3])

In [ ]:
res = []

gains = np.arange(500, 2001, 50)
for flux_gain in tqdm(gains):
    spec = do_dump_spectroscopy(config_thisrun, ds_thisrun, expt_path, aconfig_path, 
                                man_mode_no=1, dump_no=1,
                                flux_gain=flux_gain, flux_length=1)
    res.append(spec.data['avgi'])
res = np.array(res)

In [ ]:
means = res.mean(axis=1)
stds = res.std(axis=1)
plt.errorbar(gains, means, yerr=stds, marker='o')

In [ ]:
ds_thisrun.update_freq('M1-D1', 2313.2)

In [ ]:
for i in range(len(expts_to_run['stor_modes'])):
    print('Frequency for M1-S' + str(i+1) + ':', ds_thisrun.get_freq('M1-S' + str(i+1)))


### Man-coupler

In [ ]:
def get_coupler_parameters(ds_thisrun, config_thisrun, man_mode_no):
    """
    Get pulse parameters for a given storage mode. 
    Also returns prepulse and postpulse (single photon prep and meas for ge meas)

    Args:
        ds_thisrun: Dataset object for managing frequency data.
        config_thisrun: Configuration dictionary for the current run.
        man_mode_no: Manipulation mode number.
        dump_mode_no: Dump mode number.

    Returns:
        A tuple containing freq, gain, ch, prepulse, and postpulse.
    """
    stor_name = 'M' + str(man_mode_no) + '-C'
    freq = ds_thisrun.get_freq(stor_name)
    gain = ds_thisrun.get_gain(stor_name)
    pi_len = ds_thisrun.get_pi(stor_name)
    h_pi_len = ds_thisrun.get_h_pi(stor_name)
    ch = 'low' if freq < 1000 else 'high'

    from MM_dual_rail_base import MM_dual_rail_base
    mm_base_dummy = MM_dual_rail_base(config_thisrun, soccfg=soc)
    prep_man_pi = mm_base_dummy.prep_man_photon(man_mode_no)
    prepulse = mm_base_dummy.get_prepulse_creator(prep_man_pi).pulse.tolist()
    postpulse = mm_base_dummy.get_prepulse_creator(prep_man_pi[-1:-3:-1]).pulse.tolist() # for ge meas, only do f0g1 and ef pi

    prepulse_overwrite = [['multiphoton', 'g0-e0', 'pi', 0],
                            ['multiphoton', 'e0-f0', 'pi', 0],
                            ['multiphoton', 'f0-g1', 'pi', 0]
                        ]
    postpulse_overwrite = [ ['multiphoton', 'f0-g1', 'pi', 0],
                            ['multiphoton', 'e0-f0', 'pi', 0]]
    prepulse = mm_base_dummy.get_prepulse_creator(prepulse_overwrite).pulse.tolist()
    postpulse = mm_base_dummy.get_prepulse_creator(postpulse_overwrite).pulse.tolist()

    return freq, gain, pi_len, h_pi_len, ch, prepulse, postpulse


def do_coupler_spectroscopy(config_thisrun, 
                         ds_thisrun, 
                         expt_path, 
                         config_path, 
                         man_mode_no = 1, 
                         dump_no = 1,
                         flux_gain = 5000,
                         flux_length = 1):
    """
    Run the Flux Spectroscopy F0g1 Experiment.

    This function performs a flux spectroscopy experiment to measure the transition frequency
    between the f0 and g1 states of a qubit. It configures the experiment parameters, executes
    the experiment, and saves the results.

    Args:
        config_thisrun (AttrDict): Configuration dictionary for the current run.
        ds_thisrun (dataset.storage_man_swap_dataset): Dataset object for managing frequency data.
        expt_path (str): Path to save the experiment results.
        config_path (str): Path to the configuration file.
        man_mode_no (int, optional): Manipulation mode number (default is 1).
        dump_no (int, optional): Storage mode number (default is 1).

    Returns:
        FluxSpectroscopyF0g1Experiment: The experiment object containing the results.
    """
    flux_spec = meas.single_qubit.rf_flux_spectroscopy_f0g1.FluxSpectroscopyF0g1Experiment(
        soccfg=soc, path=expt_path, prefix='FluxSpectroscopyF0g1Experiment', config_file=config_path
    )

    flux_spec.cfg = AttrDict(deepcopy(config_thisrun))

    freq, gain, pi_len, h_pi_len, ch, prepulse, postpulse = get_coupler_parameters(ds_thisrun, config_thisrun,
                                                                      man_mode_no)

    # freq = 420

    flux_spec.cfg.expt = dict(
        start=freq - 50,  # Start RF frequency [MHz]
        step=0.5,  # Step size [MHz]
        expts=200,  # Number of experiments
        reps=200,  # Number of averages per point
        qubit=[0],
        flux_drive=[ch, 1, flux_gain, flux_length],  # RF flux modulation parameters [low/high (ch), freq (will be overwritten), gain, length(us)]
        prepulse=True,
        postpulse=True,
        active_reset=True,
        man_reset=False,
        storage_reset=False,
        pre_sweep_pulse= prepulse,
        post_sweep_pulse= postpulse,
    )

    flux_spec.cfg.device.readout.relax_delay = [200]  # Wait time between experiments [us]
    flux_spec.go(analyze=False, display=False, progress=True, save=True)
    return flux_spec


def update_dump_spectroscopy(flux_spec, ds_thisrun, man_mode_no = 1):
    """Update the configuration based on Flux Spectroscopy F0g1 experiment results."""
    # Update the dataset with the new frequency
    ds_thisrun.update_freq('M' + str(man_mode_no) + '-C', flux_spec.data['fit'][2])
    print(f"Updated frequency for M{man_mode_no}-C: {flux_spec.data['fit'][2]}")


In [ ]:
spec = do_coupler_spectroscopy(config_thisrun, ds_thisrun, expt_path, config_path, man_mode_no=1,
                            flux_gain=5000, flux_length=2)
# analyze_and_display_stor_spectroscopy(spec)
# update_dump_spectroscopy(spec, ds_thisrun, 1, 1)
spec.analyze(fit=True)
spec.display()

## Find Freq Chevron + Error Amp

In [ ]:
def do_sideband_general_sweep(
    config_thisrun, ds_thisrun, expt_path, config_path, freq_start, freq_stop, freq_step,
    reps=50, man_mode_no = 1, stor_mode_no = 1, start_time = 0.007, liveplotting=True):
    """Run the Sideband General Sweep Experiment."""
    from multimode_expts.sequential_experiment_classes import sidebands_class
    experiment_class = sidebands_class
    sweep_experiment_name = 'sideband_general_sweep'
    class_for_exp = experiment_class(
        soccfg=soc, path=expt_path,
        prefix=sweep_experiment_name,
        config_file=config_path,
        exp_param_file=exp_param_file,
        config_thisrun=config_thisrun,
        liveplotting=liveplotting)

    # class_for_exp.yaml_cfg = AttrDict(deepcopy(config_thisrun))
    # get pulse parameters for the given storage mode
    freq, gain, pi_len, h_pi_len, ch, prepulse, postpulse = get_storage_mode_parameters(ds_thisrun, config_thisrun, man_mode_no, stor_mode_no)
    print('gain:', gain)

    class_for_exp.loaded[sweep_experiment_name] =  dict(
        start=start_time,  # Pulse start length [us]
        step=pi_len / 5,  # Pulse step length [us]
        qubits=[0],
        expts=15, #30,
        reps=reps, #90
        rounds=1,
        freq_start=freq_start,
        freq_stop=freq_stop,
        freq_step=freq_step,
        flux_drive=[ch, freq, gain, 0.05],  # RF flux modulation parameters [low/high (ch), freq (will be overwritten), gain, length placeholder(us)]
        prepulse=True,
        postpulse=True,
        active_reset=False,
        man_reset=True,
        storage_reset=True,
        update_post_pulse_phase=[False, 1.07],
        pre_sweep_pulse= prepulse,
        post_sweep_pulse= postpulse,
    )
    class_for_exp.yaml_cfg.device.readout.relax_delay = [8000]  # Wait time between experiments [us]

    return eval('class_for_exp.run_sweep')( sweep_experiment_name = sweep_experiment_name)


def update_sideband_general_sweep(expt_path, config_thisrun, ds_thisrun, man_mode_no=1, stor_mode_no=1, update=True):
    """Update sweep data and analyze results."""
    temp_data, attrs, filename = prev_data(expt_path, prefix='sideband_general_sweep')
    print('File saved at:', filename)

    from multimode_expts.fit_display_classes import ChevronFitting
    from datetime import datetime
    chevron_analysis = ChevronFitting(
        frequencies=temp_data['freq_sweep'],
        time=temp_data['xpts'][0],
        response_matrix=temp_data['avgi'],config = config_thisrun,
    )
    chevron_analysis.analyze()
    
    current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    chevron_analysis.display_results(save_fig=True, title=f'M{man_mode_no}-S{stor_mode_no}_{current_time}')

    # Update dataset
    if not update: return chevron_analysis

    stor_name = 'M' + str(man_mode_no) + '-S' + str(stor_mode_no)
    ds_thisrun.update_freq(stor_name, chevron_analysis.results['best_frequency_period'])
    print('Updated the frequency to:', chevron_analysis.results['best_frequency_period'])

    pi_len = abs(np.pi / chevron_analysis.results['best_fit_params_period']['omega'])
    ds_thisrun.update_pi(stor_name, pi_len)
    print('Updated the pi length to:', pi_len)
    ds_thisrun.update_h_pi(stor_name, pi_len / 2)
    print('Updated the h_pi length to:', pi_len / 2)
    return chevron_analysis

In [ ]:
def do_error_amp_storage(
    config_thisrun,
    ds_thisrun,
    expt_path,
    config_path,
    reps=100,
    rounds=1,
    qubit=0,
    n_start=1,
    n_step=1,
    n_pulses=10,
    active_reset=False,
    man_reset=True,
    storage_reset=True,
    relax_delay=2500, 
    span = 1.0, # in units of either freq or gain depending on parameter_to_test
    expts = 25,
    parameter_to_test = 'frequency',
    man_mode_no=1,
    stor_mode_no=1,
    stor_is_dump=False,
):
    """
    Run the Histogram Prepulse Experiment with configurable parameters.
    """
        
    pulse_type = ['storage', f'M{man_mode_no}-{"D" if stor_is_dump else "S"}{stor_mode_no}', 'pi', 0]


    freq, gain, pi_len, h_pi_len, ch, prepulse, postpulse = get_storage_mode_parameters(ds_thisrun, config_thisrun, man_mode_no, stor_mode_no)

    if parameter_to_test == 'frequency':
        start = freq - span / 2
        step = span / (expts - 1)
    elif parameter_to_test == 'gain':
        start = int(gain - span / 2)
        step = int(span / (expts - 1))
    else:
        raise ValueError("parameter_to_test must be either 'frequency' or 'gain'.")
    
    expt_cfg = {
        'reps': reps,
        'qubit': qubit,
        'qubits': [qubit],
        'active_reset': active_reset,
        'man_reset': man_reset,
        'storage_reset': storage_reset,
        'start': start,
        'expts': expts,
        'step': step,
        'n_start': n_start,
        'n_step': n_step,
        'n_pulses': n_pulses,
        'pulse_type': pulse_type,
        'parameter_to_test': parameter_to_test,
        'rounds': rounds,
    }

    error_amp_exp = meas.single_qubit.error_amplification.ErrorAmplificationExperiment(
    soccfg=soc, path=expt_path,
      prefix='ErrorAmplificationExperiment', config_file=config_path)
    error_amp_exp.cfg = AttrDict(deepcopy(config_thisrun))
    error_amp_exp.cfg.expt = expt_cfg
    error_amp_exp.go(analyze=False, display=False, progress=True, save=True)
    return error_amp_exp

In [ ]:
expts_to_run['stor_modes'] = [1, 2, 3, 4, 5, 6, 7]
# expts_to_run['stor_modes'] = [5]

sideband_chevrons_coarse = [None] * len(expts_to_run['stor_modes'])
sideband_chevrons_fine = [None] * len(expts_to_run['stor_modes'])
error_amp_gain1 = [None] * len(expts_to_run['stor_modes'])
error_amp_freq1 = [None] * len(expts_to_run['stor_modes'])
error_amp_gain2 = [None] * len(expts_to_run['stor_modes'])
error_amp_freq2 = [None] * len(expts_to_run['stor_modes'])

In [ ]:
for i, stor_i in enumerate(expts_to_run['stor_modes']):
# for stor_i in [5]:

    stor_name = 'M1-S' + str(stor_i)
    if expts_to_run['sideband_freq_sweep']:

        # ds_thisrun.update_gain(stor_name, 9000)
        # ds_thisrun.update_freq(stor_name, 873.6)
        # ds_thisrun.update_freq(stor_name, 348.66845603890465)
        # ds_thisrun.update_pi(stor_name, abs(ds_thisrun.get_pi(stor_name)))
        # ds_thisrun.update_h_pi(stor_name, abs(ds_thisrun.get_h_pi(stor_name)))

        print(f'Running coarse sideband sweep for storage mode {stor_i}')
        freq_start = ds_thisrun.get_freq(stor_name) - 1.5
        freq_stop = ds_thisrun.get_freq(stor_name) + 1.5
        freq_step = 0.2
        # freq_start = ds_thisrun.get_freq(stor_name) - 0.2
        # freq_stop = ds_thisrun.get_freq(stor_name) + 0.2
        # freq_step = 0.1
        sideband_chevrons_coarse[i] = do_sideband_general_sweep(config_thisrun, ds_thisrun, expt_path, config_path, freq_start, freq_stop, freq_step, reps=50, man_mode_no=1, stor_mode_no=stor_i, liveplotting=False)
        update_sideband_general_sweep(expt_path, config_thisrun, ds_thisrun, man_mode_no=1, stor_mode_no=stor_i)


        print(f'Running error amplification (gain) for storage mode {stor_i}')
        error_amp_exp_gain = do_error_amp_storage(
            config_thisrun, ds_thisrun, expt_path, config_path, reps=50, rounds=1, qubit=0, span=1000, expts=50, n_start=0, n_step=3, n_pulses=15, man_mode_no=1, stor_mode_no=stor_i, stor_is_dump=False, parameter_to_test='gain')
        error_amp_gain1[i] = error_amp_exp_gain
        error_amp_exp_gain.analyze(state_fin='e')
        error_amp_exp_gain.display()
        gain_opt = int(error_amp_exp_gain.data['fit_avgi'][2])
        # Update dataset
        ds_thisrun.update_gain(stor_name, gain_opt)
        print('Updated the gain to:', gain_opt)


        print(f'Running error amplification (frequency) for storage mode {stor_i}')
        error_amp_exp_freq = do_error_amp_storage(
            config_thisrun, ds_thisrun, expt_path, config_path, reps=50, rounds=1, qubit=0, span=0.3, expts=50, n_pulses=10, man_mode_no=1, stor_mode_no=stor_i, stor_is_dump=False, parameter_to_test='frequency')
        error_amp_freq1[i] = error_amp_exp_freq
        error_amp_exp_freq.analyze(state_fin='e')
        error_amp_exp_freq.display()
        freq_opt = error_amp_exp_freq.data['fit_avgi'][2]
        # Update dataset
        ds_thisrun.update_freq(stor_name, freq_opt)
        print('Updated the frequency to:', freq_opt)


        print(f'Running error amplification (gain) ROUND 2 for storage mode {stor_i}')
        error_amp_exp_gain2 = do_error_amp_storage(
            config_thisrun, ds_thisrun, expt_path, config_path, reps=50, rounds=1, qubit=0, span=600, expts=50, n_start=1, n_step=1, n_pulses=10, man_mode_no=1, stor_mode_no=stor_i, stor_is_dump=False, parameter_to_test='gain')
        error_amp_gain2[i] = error_amp_exp_gain2
        error_amp_exp_gain2.analyze(state_fin='e')
        error_amp_exp_gain2.display()
        gain_opt = int(error_amp_exp_gain2.data['fit_avgi'][2])
        # Update dataset
        ds_thisrun.update_gain(stor_name, gain_opt)
        print('Updated the gain to:', gain_opt)



In [ ]:
# for i, stor_i in enumerate(expts_to_run['stor_modes']):
#     if stor_i != 4: continue
#     # if stor_i == 5: continue

#     stor_name = 'M1-S' + str(stor_i)
#     print(stor_name)
#     ds.update_gain(stor_name, ds_thisrun.get_gain(stor_name))
#     ds.update_freq(stor_name, ds_thisrun.get_freq(stor_name))
#     ds.update_pi(stor_name, ds_thisrun.get_pi(stor_name))
#     ds.update_h_pi(stor_name, ds_thisrun.get_h_pi(stor_name))

# Qubit characterization

In [ ]:
for i in range(len(expts_to_run['stor_modes'])):
    print('Frequency for M1-S' + str(i+1) + ':', ds_thisrun.get_freq('M1-S' + str(i+1)))

In [ ]:
# #Update sweep data and analyze results

# temp_data, attrs, filename = prev_data(expt_path, prefix='sideband_general_sweep')
# print('File saved at:', filename)

# from multimode_expts.fit_display_classes import ChevronFitting
# chevron_analysis = ChevronFitting(
#     frequencies=temp_data['freq_sweep'],
#     time=temp_data['xpts'][0],
#     response_matrix=temp_data['avgi']
# )
# chevron_analysis.analyze()

# current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
# chevron_analysis.display_results(save_fig=True,  title=f'M{1}-S{7}_{current_time}')

In [ ]:
# i = 5
# # guess_freq = 1422.66
# anls = chevrons_stor_objects[i]
# anls.analyze()
# anls.display_results(save_fig=False, 
#                     directory=autocalib_path, 
#                     title=f'M1-S{stor_i}_for_chief')
# ds_thisrun.update_freq('M1-S' + str(i+1), anls.results['best_frequency_period'])
# # or override manually 
# # ds_thisrun.update_freq('M1-S' + str(i+1), 1052.8026)

In [ ]:
# i = 5
# print(f'Running fine sideband sweep for storage mode {i + 1}')
# freq_start = ds_thisrun.get_freq('M1-S' + str(i + 1)) - 0.5
# freq_stop = ds_thisrun.get_freq('M1-S' + str(i + 1)) + 0.5
# freq_step = 0.2
# sideband_chevrons_fine = do_sideband_general_sweep(
#     config_thisrun, ds_thisrun, expt_path, config_path, freq_start, freq_stop, freq_step, 
#     man_mode_no=1, stor_mode_no=i + 1, start_time=0.1, use_flux_low=False)
# # update_sideband_general_sweep(expt_path, config_thisrun, ds_thisrun, man_mode_no=1, stor_mode_no=i + 1, use_flux_low=True)

### Manually get the frequencies from the chevrons

In [ ]:
# file_list = [34,36,38,40,42,44,46]
# chevrons_stor_objects = [None]* len(file_list)
# name = '_sideband_general_sweep.h5'
# # name = '_storage_sideband_sweep.h5'
# for idx, file_no in enumerate(file_list):
#     full_name = str(file_no).zfill(5)+name
#     expt_path_mod = r"H:\Shared drives\SLab\Multimode\experiment\250505_craqm\data"
#     temp_data, attrs, filename = prev_data(expt_path_mod, full_name)  # ef

#     from multimode_expts.fit_display_classes import ChevronFitting
#     chevron_analysis = ChevronFitting(
#         frequencies=temp_data['freq_sweep'],
#         time=temp_data['xpts'][0],
#         response_matrix=temp_data['avgi']
#     )
#     chevrons_stor_objects[idx] = chevron_analysis

In [ ]:
# i = 6
# # guess_freq = 1422.66
# anls = chevrons_stor_objects[i]
# anls.analyze()
# anls.display_results(save_fig=False, 
#                     directory=autocalib_path, 
#                     title=f'M1-S{i+1}')
#                 #  hlines = [guess_freq])
# ds_thisrun.update_freq('M1-S' + str(i+1), anls.results['best_frequency_period'])
# # or override manually 
# # ds_thisrun.update_freq('M1-S' + str(i+1), 1052.8026)

In [ ]:
# ds_thisrun.df

In [ ]:

# ds_thisrun.update_freq('M1-S' + str(i), guess_freq)

In [ ]:
# ds_thisrun.get_freq('M1-S3')

## Sideband General

In [ ]:
def do_sideband_general(config_thisrun, expt_path, config_path, man_mode_no = 1,stor_mode_no=1):
    """Run the Sideband General Experiment."""
    
    sideband_general = meas.single_qubit.sideband_general.SidebandGeneralExperiment(
        soccfg=soc, path=expt_path, prefix='SidebandGeneralExperiment', config_file=config_path
    )

    sideband_general.cfg = AttrDict(deepcopy(config_thisrun))

    # Sideband general experiment parameters
    freq, gain, pi_len, h_pi_len, ch, prepulse, postpulse = get_storage_mode_parameters(ds_thisrun, config_thisrun, man_mode_no, stor_mode_no)

    sideband_general.cfg.expt = dict(
        start=0.007,  # Pulse start length [us]
        step=0.05,  # Pulse step length [us]
        qubits=[0],
        expts=100,
        reps=200,
        rounds=1,
        flux_drive=[ch, freq, gain],  # RF flux modulation
        prepulse=True,
        postpulse=True,
        active_reset=False,
        man_reset=True,
        storage_reset=True,
        update_post_pulse_phase=[False, 1.07],
        pre_sweep_pulse=prepulse,
        post_sweep_pulse=postpulse,
    )

    sideband_general.cfg.device.readout.relax_delay = [8000]  # Wait time between experiments [us]
    sideband_general.go(analyze=False, display=False, progress=True, save=True)
    from multimode_expts.fit_display_classes import LengthRabiFitting
    sideband_analysis = LengthRabiFitting(sideband_general.data, config=sideband_general.cfg)
    sideband_analysis.analyze()
    sideband_analysis.display(title_str='Sideband General', save_fig=True)

    return sideband_analysis


def update_sideband_general(sideband_general, config_thisrun, ds, man_mode_no=1, stor_mode_no=1):
    """Update the configuration and dataset based on Sideband General experiment results."""
    # Analyze the data
    

    # Update dataset
    pi_length = sideband_general.results['pi_length']
    pi2_length = sideband_general.results['pi2_length']
    gain = sideband_general.cfg.expt['flux_drive'][2]
    freq = sideband_general.cfg.expt['flux_drive'][1]
    ds.update_all('M' + str(man_mode_no) + '-S' + str(stor_mode_no), freq, np.nan, pi_length, pi2_length, gain)
    print(f'Updated configuration and dataset: pi_length={pi_length}, pi2_length={pi2_length}, gain={gain}')



In [ ]:
len_rabi_storages = [None]* len(expts_to_run['stor_modes'])
if expts_to_run['sideband_length_rabi']:
    for man_mode_no in expts_to_run['man_modes']:
        for stor_mode_no in expts_to_run['stor_modes']:
            print(f'Running sideband general for manipulation mode {man_mode_no} and storage mode {stor_mode_no}')
            len_rabi_storages[stor_mode_no-1] = do_sideband_general(config_thisrun, expt_path, config_path, man_mode_no, stor_mode_no)
            update_sideband_general(len_rabi_storages[stor_mode_no-1], config_thisrun, ds_thisrun, man_mode_no, stor_mode_no)


In [ ]:
def do_error_amplification(
    config_thisrun,
    ds_thisrun,
    expt_path,
    config_path,
    reps=100,
    rounds=1,
    qubit=0,
    n_pulses=10,
    active_reset=False,
    man_reset=True,
    storage_reset=True,
    relax_delay=2500, 
    start = 0,
    expts = 10,
    step = 100,
    parameter_to_test = 'gain',
    pulse_type=['qubit', 'ge', 'pi', 0],
):
    """
    Run the Histogram Prepulse Experiment with configurable parameters.
    """
        
    expt_cfg = {
        'reps': reps,
        'qubit': qubit,
        'qubits': [qubit],
        'active_reset': active_reset,
        'man_reset': man_reset,
        'storage_reset': storage_reset,
        'start': start,
        'expts': expts,
        'step': step,
        'n_pulses': n_pulses,
        'pulse_type': pulse_type,
        'parameter_to_test': parameter_to_test,
        'rounds': rounds,

    }
    error_amp_exp = meas.single_qubit.error_amplification.ErrorAmplificationExperiment(
    soccfg=soc, path=expt_path,
      prefix='ErrorAmplificationExperiment', config_file=config_path)
    error_amp_exp.cfg = AttrDict(deepcopy(config_thisrun))
    error_amp_exp.cfg.expt = expt_cfg
    error_amp_exp.go(analyze=False, display=False, progress=True, save=True)
    return error_amp_exp

## Randomized Benchmarking

In [ ]:
len([1, 5, 10, 15, 25, 50, 75, 100, 150, 200, 250, 400, 500, 600, 700, 850, 1000, 1250, 1500])

In [ ]:
# config_thisrun.device.storage.ramp_sigma
# man_mode_no = 1
# stor_mode_no = 7
# req = ds_thisrun.get_freq('M' + str(man_mode_no) + '-S' + str(stor_mode_no))
# gain = ds_thisrun.get_gain('M' + str(man_mode_no) + '-S' + str(stor_mode_no))
# hpi_length = ds_thisrun.get_h_pi('M' + str(man_mode_no) + '-S' + str(stor_mode_no))
# print("ramp_sigma:", config_thisrun.device.storage.ramp_sigma)
# print("man_mode_no:", man_mode_no)
# print("stor_mode_no:", stor_mode_no)
# print("freq:", req)
# print("gain:", gain)
# print("hpi_length:", hpi_length)

In [ ]:

def do_single_beam_splitter_rb_postselection_sweep_depth(config_thisrun, ds_thisrun, expt_path, config_path, exp_param_file, man_mode_no=1, stor_mode_no=1,
                                                         prev_data=None):
    """
    Run the SingleBeamSplitterRBPostSelection_sweep_depth experiment.
    """
    from multimode_expts.sequential_experiment_classes import MM_DualRailRB
    experiment_class = MM_DualRailRB
    sweep_experiment_name = 'SingleBeamSplitterRBPostSelection_sweep_depth'
    class_for_exp = experiment_class(
        soccfg=soc, path=expt_path, prefix=sweep_experiment_name, config_file=config_path, exp_param_file=exp_param_file, 
        prev_data=prev_data
    )

    class_for_exp.yaml_cfg = AttrDict(deepcopy(config_thisrun))
    # Customize bs_para for the given manipulation and storage mode using ds_thisrun directly
    freq = ds_thisrun.get_freq('M' + str(man_mode_no) + '-S' + str(stor_mode_no))
    gain = ds_thisrun.get_gain('M' + str(man_mode_no) + '-S' + str(stor_mode_no))
    hpi_length = ds_thisrun.get_h_pi('M' + str(man_mode_no) + '-S' + str(stor_mode_no))
    bs_para = [freq, gain, hpi_length, config_thisrun.device.storage.ramp_sigma]
    print('Beam splitter parameters:', bs_para)
    # Optionally, set up experiment parameters here if needed, e.g.:
    # class_for_exp.loaded[sweep_experiment_name] = dict(...)
    class_for_exp.loaded[sweep_experiment_name] = dict(
        depth_list=[1, 5, 10, 15, 25, 50, 75, 100, 150, 200, 250, 400, 500, 600, 700, 850, 1000, 1250, 1500, 2000],  # RB sequence depth list
        reps_list = [],
        qubits=[0],
        reps=0,  # doesn't matter

        single_shot_bef_expt=False,  # single shot before experiment
        singleshot_reps=2000,        # single shot measurement repetitions
        span=1000,                   # single shot plot span

        active_reset=False,          # for single shot post selection
        man_reset=True,              # for single shot post selection
        storage_reset=True,          # for single shot post selection
        threshold=None,              # for single shot post selection
        readout_per_round=4,         # for single shot post selection

        rb_active_reset=False,
        rb_man_reset=True,
        rb_storage_reset=True,
        rb_reps=1000,
        gates_per_wait=100000,       # ????
        parity_meas=True,            # If parity measurement is used, set to True; if False the reset arguments below should be false as well
        reset_qubit_after_parity=False,  # True # resetting via second parity str 
        reset_qubit_via_active_reset_after_first_meas=False,  # resetting via active reset after first parity str; the other reset should be false

        rounds=1,                    # always set to 1
        variations=10,                # number of different sequences
        rb_depth=10,                 # rb sequence depth
        IRB_gate_no=-1,              # IRB gate number, -1 means not using
        postselection_delay=2.0,     # in us, gap between two readout pulses
        bs_repeat=1,
        sync=False,
        setup=False,

        bs_para=bs_para,  # at 96  # beam splitter parameters [[frequency], [gain], [length (us)], [sigma]]
        prepulse=False,
        postpulse=False,
        f0g1_offset=0,               # offset phase in deg as a result of f0g1 prepulse/postpulse

        pre_sweep_pulse=[[None]],    # Gate based; prep f0g1 is done automatically ; RAM state prep is also automatic 
        ram_prepulse=[False, 6, [1], 1],  # [True/False, number of storage modes to be populated, [idx of modes to be skipped], variations]
        ram_prepulse_strs=None       # see SingleBeamSplitterRBPostSelection_sweep_depth_and_ram
    )
    

    class_for_exp.yaml_cfg.device.readout.relax_delay = [8000]  # Example, adjust as needed

    prefix, dir_path =eval('class_for_exp.run_sweep')(sweep_experiment_name=sweep_experiment_name)
    print('File saved at:', prefix)
    print('Directory path:', dir_path)

    # from multimode_expts.fit_display_classes import MM_DualRailRBFitting
    # rb_analysis = MM_DualRailRBFitting(
    #     filename=None,
    #     file_prefix=prefix,
    #     config=yaml_cfg,
    #     expt_path=expt_path,
    #     title=f'M{man_mode_no}-S{stor_mode_no}',
    #     prev_data=prev_data,
    #     dir_path=dir_path
    # )
    # rb_analysis.show_rb(save_fig=True)
    # return rb_analysis


In [ ]:
if expts_to_run['RB']:
    storage_rbs = [None] * len(expts_to_run['stor_modes'])
    for i in range(len(expts_to_run['stor_modes'])):
        print(f'Running storage RB postselection sweep depth for storage mode {i + 1}')
        storage_rbs[i] = do_single_beam_splitter_rb_postselection_sweep_depth(
            config_thisrun,
            ds_thisrun,
            expt_path,
            config_path,
            exp_param_file=exp_param_file,
            man_mode_no=1,
            stor_mode_no=i + 1
        )

In [ ]:
# from multimode_expts.fit_display_classes import MM_DualRailRBFitting
# dir_no = np.arange(72, 80, 1 )
# for idx, dir_no in enumerate(dir_no):
#     prefix = f"SingleBeamSplitterRBPostSelection_sweep_depth"
#     dir_name = f"{str(dir_no).zfill(5)}_SingleBeamSplitterRBPostSelection_sweep_depth" 
#     dir_path = r"H:\\\\Shared drives\\\\SLab\\\\Multimode\\\\experiment\\\\250505_craqm\\\\data\\\\" + dir_name
#     filepath = dir_path
#     # Initialize RB analysis
#     rb_analysis = MM_DualRailRBFitting(
#         filename=None,
#         file_prefix=prefix,
#         config=yaml_cfg,
#         expt_path=expt_path,
#         title=f"M1-S{idx + 1} RB Analysis",
#         prev_data=prev_data,
#         dir_path=filepath
#     )
#     rb_analysis.show_rb(save_fig=True)



In [ ]:
dir_no = 78
dir_name = f"{str(dir_no).zfill(5)}_SingleBeamSplitterRBPostSelection_sweep_depth" 
dir_path = r"H:\\\\Shared drives\\\\SLab\\\\Multimode\\\\experiment\\\\250505_craqm\\\\data\\\\" + dir_name
temp_data, attrs, filename = prev_data(dir_path, prefix='SingleBeamSplitterRBPostSelection_sweep_depth')

In [ ]:
atrrs = AttrDict(attrs)
attrs['config']['device']['qubit']['f_ge']

In [ ]:
# storage_rb

In [ ]:
expt_test = Experiment(
            path=expt_path,
            prefix="yoyoyo",
            config_file=config_path,
        )

In [ ]:
expt_test.data = {}
f = expt_test.save_data()

In [ ]:
if expts_to_run['RB']:
    filename = expt_test.fname
    #create a directory with the filename but no .h5 extension
    import os
    directory = filename
    if directory.lower().endswith('.h5'):
        directory = directory[:-3]
    if not os.path.exists(directory):
        # Only create the directory if it is not the same as the filename (i.e., filename is not a directory itself)
        # Make sure the directory name does not have a .h5 extension
        os.makedirs(directory)

In [ ]:
# directory
# filename_only = os.path.basename(expt_test.fname)
# filename_only

In [ ]:
if expts_to_run['RB']:
    from multimode_expts.fit_display_classes import MM_DualRailRBFitting
    rb_analysis = MM_DualRailRBFitting(file_prefix = "SingleBeamSplitterRBPostSelection_sweep_depth", 
                                    config=config_thisrun, expt_path=expt_path, title='M1_S1', 
                                    prev_data= prev_data)
    rb_analysis.show_rb()

In [ ]:
if expts_to_run['RB']:
    temp_data, attrs, filename = prev_data(expt_path, '00036_SingleBeamSplitterRBPostSelection_sweep_depth.h5')

In [ ]:
attrs['config'].keys()

In [ ]:
temp_data.keys()

In [ ]:
num_entries = len(temp_data['Idata'])
print(f"Number of entries in 'Idata': {num_entries}")

In [ ]:
temp_data['sequences']

# Shock TLS Function

In [ ]:
from multimode_expts.experiments.single_qubit.pulse_probe_f0g1_spectroscopy import PulseProbeF0g1SpectroscopyExperiment

def do_pulse_probe_f0g1_spectroscopy(config_thisrun, expt_path, config_path, 
                                     start=2007, step=0.02, expts=300, reps=100,
                                       rounds=1, length=1, gain=5000, pulse_type='gaussian',
                                         qubit_f=True, qubits=[0], prepulse=False, pre_sweep_pulse=None):
    """
    Run the PulseProbeF0g1SpectroscopyExperiment with specified parameters.
    """
    
    expt = PulseProbeF0g1SpectroscopyExperiment(
        soccfg=soc, path=expt_path, prefix='PulseProbeF0g1SpectroscopyExperiment', config_file=config_path
    )
    expt.cfg = AttrDict(deepcopy(config_thisrun))
    expt.cfg.expt = dict(
        start=start,
        step=step,
        expts=expts,
        reps=reps,
        rounds=rounds,
        length=length,
        gain=gain,
        pulse_type=pulse_type,
        qubit_f=qubit_f,
        qubits=qubits,
        prepulse=prepulse,
        pre_sweep_pulse=pre_sweep_pulse
    )
    expt.cfg.device.readout.relax_delay = [5]  # Wait time between experiments [us]
    expt.go(analyze=True, display=True, progress=True, save=True)
    return expt

In [ ]:
# t2ramsey_ge_check = None
# while True: 
#     #close previous plots 
#     # import matplotlib.pyplot as plt
#     #each iteration is 5 minutes
#     # do_pulse_probe_f0g1_spectroscopy(
#     #     config_thisrun, expt_path, config_path,
#     #     start=3300, step=0.04, expts=10000, reps=100,
#     #     rounds=5, length=50, gain=30000, pulse_type='gaussian',
#     #     qubit_f=False, qubits=[0], prepulse=False, pre_sweep_pulse=None
#     # )

#     do_pulse_probe_ge(config_thisrun, start = 3300, 
#                       step = 0.04, expts = 10000, reps = 100, rounds = 5,
#                       length = 50, gain = 30000)
#     from IPython.display import clear_output
#     # from multimode_expts.fit_display_classes import SidebandFitting
#     clear_output(wait=True)
#     plt.close('all')  # Close all existing figures
#     t2ramsey_ge_check = do_t2_ramsey_ge(config_thisrun, expt_path, config_path)
#     t2ramsey_ge_check.analyze()
#     t2ramsey_ge_check.display(title_str='T2_ge_TLS')


In [ ]:
from multimode_expts.MM_base import MMAveragerProgram
from multimode_expts.experiments.qsim.utils import ensure_list_in_cfg

class ShockTLSProgram(MMAveragerProgram):
    def initialize(self):
        self.MM_base_initialize()
        
        
    def body(self):
        self.reset_and_sync()

        self.setup_and_pulse(
            ch=0,
            style='const',
            freq=self.freq2reg(1918, gen_ch=0),
            length=50,
            phase=0,
            gain=30000,
        )
    def update(self):
        print('Updating frequency...')
        print('Current frequency (MHz):', self.reg2freq(self.r_freq2, gen_ch=0))
        print('Frequency step (MHz):', self.f_step)
        self.mathi(self.q_rp, self.r_freq2, self.r_freq2, '+', self.f_step) # update frequency list index


class ShockTLSExperiment(Experiment):
    def acquire(self, progress):
        ensure_list_in_cfg(self.cfg)
        
        self.prog = ShockTLSProgram(soccfg=self.soccfg, cfg=self.cfg)
        self.prog.acquire(self.im[self.cfg.aliases.soc],
                         threshold=None,
                         load_pulses=True,
                         progress=progress,
                         debug=False)
    

### ShockTLS !!!

In [ ]:
shockTLS_bool = False # Set to FALSE to avoid running!!!!!!
if shockTLS_bool:    
    scktls = ShockTLSExperiment(soccfg=soc, path=expt_path, prefix='shockTLS',
                            config_file=config_path)
    scktls.cfg = AttrDict(deepcopy(config_thisrun))

    scktls.cfg.expt = dict(expts=2000, reps=2000, rounds=100000, qubit=[0])

    scktls.cfg.device.readout.relax_delay = [50] # Wait time between experiments [us]
    for kk in range(50):
        scktls.go(analyze=False, display=False, progress=True, save=False)

# Update Config and Dataset

In [ ]:
station.handle_config_update(write_to_file=True)

# Tuning Yoko

In [ ]:
from slab.instruments import YokogawaGS200
dcflux = YokogawaGS200(address="10.108.30.37")
dcflux.set_output(True)
dcflux.set_mode('current')

In [ ]:
# dcflux.set_volt(0.0001)
# dcflux.get_volt()

In [ ]:
dcflux.set_current(0.000001)
dcflux.get_current()

In [ ]:
# dcflux.set_output(True)
dcflux.set_output(False)

### Current sweep to check coupler alive

In [ ]:
# if expts_to_run['res_spec']:  
# 7387.7, 7493.3, 7524.9, 7588.2?, 7599.5, 7914.2, 7494.3
rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, 
                    frequency = 7588, span = 3, reps = 400, expts = 300, gain=0.5, length = 20, final_delay=250)
spec_analysis = meas.SpectroscopyFitting(data=rspec.data, station = station)
spec_analysis.analyze()
spec_analysis.display(title = 'Readout Spectroscopy')

In [ ]:
import os
import time
from datetime import datetime

# current = np.arange(-0.3e-3, 0.3e-3, 0.9e-5) #fine sweep
# current = np.arange(-0.2e-3, 0.2e-3, 0.9e-5) #fast sweep
# current = np.arange(0, 0.2e-3, 0.1e-3) #very fast sweep

current = np.arange(-0.22e-3, 0.05e-3, 0.3e-5)
dcflux.set_mode('current')

# f_tests = [7387.7, 7493.3, 7524.9, 7588.2, 7599.5, 7914.2, 7494.3]

# f_tests = [7493.863, 7525.123]
# f_tests = [7588.084]
f_tests = [7493.863] #7599.607] #[7387.69491, 7599.607]

for freq in f_tests:
    res_data_amp = []
    res_data_avgi = []
    res_data_avgq = []
    freq_arr = []
    for cur in current:
        dcflux.set_current(cur)
        dcflux.set_output(True)
        print(f'Set current to {cur*1e3} mA')
        # rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, 
        #                     frequency = freq, span = 3, reps = 10, expts = 10, gain=0.1, final_delay=250)
        
        # rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, 
        #             frequency = freq, span = 2.5, reps = 600, expts = 3500, gain=0.7, length=20.0, final_delay=250)
    
        rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, 
            frequency = freq, span = 3, reps = 400, expts = 400, gain=0.5, final_delay=250) #for 7493, 7524 mode
        # rspec = do_res_spec(config_thisrun=config_thisrun, analyze_and_display=False, 
        #         frequency = freq, span = 2, reps = 500, expts = 1200, gain=0.5, final_delay=250) #for 7588 mode
    
        res_data_amp.append(rspec.data['amps'])
        res_data_avgi.append(rspec.data['avgi'])
        res_data_avgq.append(rspec.data['avgq'])
        freq_arr.append(rspec.data['xpts'])
        time.sleep(0.1)  # wait for 100 ms before setting the next value
        
    res_data_amp = np.array(res_data_amp)
    res_data_avgi = np.array(res_data_avgi)
    res_data_avgq = np.array(res_data_avgq)
    freq_array = np.array(freq_arr[0])  

    X, Y = np.meshgrid(freq_array, current*1e3)
    Z_amp = res_data_amp.reshape(len(current), len(freq_array))
    Z_i = res_data_avgi.reshape(len(current), len(freq_array))
    Z_q = res_data_avgq.reshape(len(current), len(freq_array))

    fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 12))
    c1 = axes[0].pcolor(X, Y, Z_amp)
    c2 = axes[1].pcolor(X, Y, Z_i)
    c3 = axes[2].pcolor(X, Y, Z_q)
    axes[0].set_title(f"Colormap of IQ vs Frequency (center: {freq} MHz) and Current")
    axes[2].set_xlabel("Frequency (MHz)")

    clabels = [c1, c2, c3]
    ylabels = ["Amplitude [ADC]", "I [ADC]", "Q [ADC]"]

    for i in range(3):
        axes[i].set_ylabel(ylabels[i])
        fig.colorbar(clabels[i], ax=axes[i])
    plt.savefig(f"{datetime.now()}_res_spec_vs_current_{freq}MHz.png".replace(' ', '_').replace(':', ''))
    print(f'plot saved at: {os.getcwd()}')
    plt.show()
    
dcflux.set_output(False) # Turn off the output after the sweep is done!! prevent heating up

# time.sleep(3600) #rest an hour to cooldown fridge after the sweep before next experiment

In [ ]:
X, Y = np.meshgrid(freq_array, current*1e3)
Z_amp = res_data_amp.reshape(len(current), len(freq_array))
Z_i = res_data_avgi.reshape(len(current), len(freq_array))
Z_q = res_data_avgq.reshape(len(current), len(freq_array))

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 12))
c1 = axes[0].pcolor(X, Y, Z_amp)
c2 = axes[1].pcolor(X, Y, Z_i)
c3 = axes[2].pcolor(X, Y, Z_q)
axes[0].set_title("Colormap of values vs Frequency and Current")
axes[2].set_xlabel("Frequency (MHz)")

clabels = [c1, c2, c3]
ylabels = ["Amplitude [ADC]", "I [ADC]", "Q [ADC]"]

for i in range(3):
    axes[i].set_ylabel(ylabels[i])
    # axes[i].color(0, 0.3)
    fig.colorbar(clabels[i], ax=axes[i])
plt.show()